In [ ]:
# MO
# Models are trained (already trained and checkpoints were saved) under different MO cross-asset-influence terms (rho)
# and simulations are run in each model's respective Environment 
# Respective Environment : (with the same set up parameters used in training environments (NOT in a single environment) . 
# Here one PnL simulation (evaluation) at a time is being run. This code can be optimized to run all simulations at once. 
# Somewhere below, you can find and example 

import os
import sys
import json
import numpy as np
import matplotlib.pyplot as plt

from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import VecMonitor

# ============================================================
# 0. PROJECT PATH
# ============================================================
import os
PROJECT_ROOT = os.environ.get("MBT_PROJECT_ROOT")
if not PROJECT_ROOT:
    PROJECT_ROOT = os.getcwd()
    while not os.path.isdir(os.path.join(PROJECT_ROOT, "mbt_gym")) and os.path.dirname(PROJECT_ROOT) != PROJECT_ROOT:
        PROJECT_ROOT = os.path.dirname(PROJECT_ROOT)

if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

import mbt_gym  # noqa: F401

# ============================================================
# 1. IMPORT YOUR MODULES
# ============================================================
from environment_set_up.N_env_setup import make_sb_env, make_eval_sb_env
from environment_set_up.N_env_builders import get_cj_env, set_env_globals

# ============================================================
# 2. PATHS
#    Replace these with your NEW trained run paths
# ============================================================
checkpoint_path = os.path.join(PROJECT_ROOT, "N_SB_models/PPO_Checkpoints_2Assets/PPO_2Assets_Trial12/PPO_2Assets_Trial12_100M.zip")
best_model_path = os.path.join(PROJECT_ROOT, "N_SB_models/PPO_Best_2Assets/PPO_2Assets_Trial1/best_model.zip")

use_best_model = False
model_path = best_model_path if use_best_model else checkpoint_path

# ============================================================
# 3. INFER NAMES FROM MODEL PATH
# ============================================================
def infer_trial_folder_from_model_path(model_path: str) -> str:
    """
    Examples:
      checkpoint: ...\\PPO_2Assets_Trial1\\PPO_2Assets_Trial1_100M.zip -> PPO_2Assets_Trial1
      best model: ...\\PPO_2Assets_Trial1\\best_model.zip              -> PPO_2Assets_Trial1
    """
    return os.path.basename(os.path.dirname(model_path))


def infer_assets_tag_from_trial_folder(trial_folder: str) -> str:
    """
    PPO_2Assets_Trial1 -> 2Assets
    PPO_3Assets_Trial7 -> 3Assets
    """
    parts = trial_folder.split("_")
    if len(parts) >= 2:
        return parts[1]
    raise ValueError(f"Could not infer assets_tag from trial folder: {trial_folder}")


def infer_run_tag(model_path: str, use_best_model: bool) -> str:
    """
    checkpoint -> filename without extension
    best_model -> <trial_folder>_Best_Model
    """
    if use_best_model:
        trial_folder = infer_trial_folder_from_model_path(model_path)
        return f"{trial_folder}_Best_Model"
    return os.path.splitext(os.path.basename(model_path))[0]


trial_folder = infer_trial_folder_from_model_path(model_path)
assets_tag = infer_assets_tag_from_trial_folder(trial_folder)
run_tag = infer_run_tag(model_path, use_best_model)

print("trial_folder:", trial_folder)
print("assets_tag  :", assets_tag)
print("run_tag     :", run_tag)

# ============================================================
# 4. SAVE PATHS
# ============================================================
fig_root = os.path.join(PROJECT_ROOT, f"N_figures/{assets_tag}/PnL_Graph")
data_root = os.path.join(PROJECT_ROOT, f"N_figures/{assets_tag}/PnL_Data")

fig_dir = os.path.join(fig_root, trial_folder)
data_dir = os.path.join(data_root, trial_folder)

os.makedirs(fig_dir, exist_ok=True)
os.makedirs(data_dir, exist_ok=True)

print("\nSaving under:")
print("  fig_dir :", fig_dir)
print("  data_dir:", data_dir)

# ============================================================
# 5. ENV SETTINGS
# ============================================================
terminal_time = 1.0
n_steps = 100
num_assets = int(assets_tag.replace("Assets", ""))
num_trajectories = 1000

# record-keeping / environment seeds
train_seed = 123
network_seed = 123   # only metadata here; not used by PPO.load
eval_seed = 456

# ============================================================
# 6. EVALUATION REWARD SETTINGS
# You said:
#   - running inventory penalty = 0
#   - terminal penalty same as original paper/training setup
# So:
#   phi = 0.0
#   alpha = 0.2
# ============================================================
eval_phi = 0.0
eval_alpha = 0.2

n_eval_episodes = 100000
deterministic = True

# ============================================================
# 7. DYNAMICS SETTINGS
#    IMPORTANT:
#    These must match the dynamics used for the checkpoint you load.
#    Update these if your new run used different values.
# ============================================================

# --- Override only MO cross-asset influence (as in your training notebook) ---
cross_asset_influence = [[0.0, 9.0],
                         [9.0, 0.0]]

# Optional LOB cross-asset influence override.
# Keep None if you want builder defaults.
c_cross_asset_influence =[[0.0, 0.0],
                         [0.0, 0.0]]  # None

# Optional additional overrides if your checkpoint used non-default dynamics.
# Keep as None to use defaults from N_env_builders.py
baseline_arrival_rate = None
mean_reversion_speed = None
self_jump_size = None
mutual_jump_size = None
synchrony_factor = None

c_baseline_depth = None
c_mean_reversion_speed = None
c_self_jump_size = None
c_mutual_jump_size = None
c_synchrony_factor = None

sigma = None
initial_price = None
initial_inventory = None  # keep None unless you want to force it

# Pack env kwargs once so train/eval envs are guaranteed consistent
env_kwargs = dict(
    sigma=sigma,
    initial_price=initial_price,
    baseline_arrival_rate=baseline_arrival_rate,
    cross_asset_influence=cross_asset_influence,
    mean_reversion_speed=mean_reversion_speed,
    self_jump_size=self_jump_size,
    mutual_jump_size=mutual_jump_size,
    synchrony_factor=synchrony_factor,
    c_baseline_depth=c_baseline_depth,
    c_cross_asset_influence=c_cross_asset_influence,
    c_mean_reversion_speed=c_mean_reversion_speed,
    c_self_jump_size=c_self_jump_size,
    c_mutual_jump_size=c_mutual_jump_size,
    c_synchrony_factor=c_synchrony_factor,
)

# Remove None-valued items so only intended overrides are passed
env_kwargs = {k: v for k, v in env_kwargs.items() if v is not None}

print("\nEnvironment overrides being used:")
for k, v in env_kwargs.items():
    print(f"  {k}: {v}")

# ============================================================
# 8. REBUILD ENVIRONMENT
#    set_env_globals controls terminal_time, n_steps, reward params, train_seed
#    cross_asset_influence is passed through make_sb_env/make_eval_sb_env
# ============================================================
set_env_globals(
    terminal_time=terminal_time,
    n_steps=n_steps,
    phi=eval_phi,
    alpha=eval_alpha,
    train_seed=train_seed,
)

train_bundle = make_sb_env(
    get_env_fn=get_cj_env,
    num_trajectories=num_trajectories,
    num_assets=num_assets,
    initial_inventory=initial_inventory,
    do_reward_scaling=False,
    env_seed=train_seed,
    debug=True,
    **env_kwargs,
)

eval_bundle = make_eval_sb_env(
    get_env_fn=get_cj_env,
    train_bundle=train_bundle,
    eval_seed=eval_seed,
    initial_inventory=initial_inventory,
    debug=True,
    **env_kwargs,
)

sb_eval_env = VecMonitor(eval_bundle.sb_eval_env)

# ============================================================
# 9. LOAD MODEL
# ============================================================
def load_model_robust(path, env):
    try:
        model = PPO.load(path, env=env, device="cpu")
        print("\nModel loaded successfully with standard PPO.load().")
        return model
    except Exception as e:
        print("\nStandard PPO.load() failed:")
        print(e)
        print("\nRetrying with custom_objects...")

        model = PPO.load(
            path,
            env=env,
            device="cpu",
            custom_objects={
                "learning_rate": 0.0,
                "lr_schedule": lambda _: 0.0,
                "clip_range": lambda _: 0.2,
            },
        )
        print("Model loaded successfully with custom_objects.")
        return model


model = load_model_robust(model_path, sb_eval_env)

# ============================================================
# 10. COLLECT EPISODE REWARDS
# ============================================================
def collect_episode_rewards_from_vecenv(model, vec_env, n_episodes=1000, deterministic=True):
    rewards = []
    obs = vec_env.reset()
    next_report = 500

    while len(rewards) < n_episodes:
        actions, _ = model.predict(obs, deterministic=deterministic)
        obs, step_rewards, dones, infos = vec_env.step(actions)

        for done, info in zip(dones, infos):
            if done:
                if "episode" in info and "r" in info["episode"]:
                    rewards.append(float(info["episode"]["r"]))
                else:
                    raise RuntimeError(
                        "Episode finished but info['episode']['r'] not found. "
                        "Check VecMonitor wrapping."
                    )

        while len(rewards) >= next_report:
            print(f"Collected {next_report} / {n_episodes} completed episodes")
            next_report += 500

    return np.array(rewards[:n_episodes], dtype=float)


episode_rewards = collect_episode_rewards_from_vecenv(
    model=model,
    vec_env=sb_eval_env,
    n_episodes=n_eval_episodes,
    deterministic=deterministic,
)

# ============================================================
# 11. SUMMARY
# ============================================================
def summarize(x):
    return {
        "n_episodes": int(len(x)),
        "mean": float(np.mean(x)),
        "variance_sample": float(np.var(x, ddof=1)),
        "std_sample": float(np.std(x, ddof=1)),
        "min": float(np.min(x)),
        "max": float(np.max(x)),
        "q01": float(np.quantile(x, 0.01)),
        "q05": float(np.quantile(x, 0.05)),
        "q25": float(np.quantile(x, 0.25)),
        "q50": float(np.quantile(x, 0.50)),
        "q75": float(np.quantile(x, 0.75)),
        "q95": float(np.quantile(x, 0.95)),
        "q99": float(np.quantile(x, 0.99)),
    }


summary = summarize(episode_rewards)

print("\n===== TERMINAL CUMULATIVE REWARD SUMMARY =====")
for k, v in summary.items():
    if isinstance(v, float):
        print(f"{k:>18}: {v:.8f}")
    else:
        print(f"{k:>18}: {v}")

# ============================================================
# 12. SAVE RAW DATA
# ============================================================
base_name = f"{run_tag}_PnL"

rewards_txt_path = os.path.join(data_dir, f"{base_name}_episode_rewards.txt")
rewards_csv_path = os.path.join(data_dir, f"{base_name}_episode_rewards.csv")
summary_txt_path = os.path.join(data_dir, f"{base_name}_summary.txt")
summary_json_path = os.path.join(data_dir, f"{base_name}_summary.json")

# raw rewards
np.savetxt(rewards_txt_path, episode_rewards, fmt="%.10f")
np.savetxt(
    rewards_csv_path,
    episode_rewards,
    delimiter=",",
    header="terminal_cumulative_reward",
    comments=""
)

# ============================================================
# 13. SAVE SUMMARY / METADATA
# ============================================================
meta = {
    "run_tag": run_tag,
    "trial_folder": trial_folder,
    "assets_tag": assets_tag,
    "use_best_model": bool(use_best_model),
    "source_model_path": model_path,
    "figure_dir": fig_dir,
    "data_dir": data_dir,
    "environment": {
        "terminal_time": float(terminal_time),
        "n_steps": int(n_steps),
        "num_assets": int(num_assets),
        "num_trajectories_parallel": int(num_trajectories),
        "initial_inventory_override": initial_inventory,
    },
    "reward_settings": {
        "phi_running_inventory_penalty": float(eval_phi),
        "alpha_terminal_inventory_penalty": float(eval_alpha),
    },
    "dynamics_overrides": {
        "cross_asset_influence": cross_asset_influence,
        "c_cross_asset_influence": c_cross_asset_influence,
        "sigma": sigma.tolist() if isinstance(sigma, np.ndarray) else sigma,
        "initial_price": initial_price.tolist() if isinstance(initial_price, np.ndarray) else initial_price,
        "baseline_arrival_rate": baseline_arrival_rate.tolist() if isinstance(baseline_arrival_rate, np.ndarray) else baseline_arrival_rate,
        "mean_reversion_speed": mean_reversion_speed.tolist() if isinstance(mean_reversion_speed, np.ndarray) else mean_reversion_speed,
        "self_jump_size": self_jump_size.tolist() if isinstance(self_jump_size, np.ndarray) else self_jump_size,
        "mutual_jump_size": mutual_jump_size.tolist() if isinstance(mutual_jump_size, np.ndarray) else mutual_jump_size,
        "synchrony_factor": synchrony_factor.tolist() if isinstance(synchrony_factor, np.ndarray) else synchrony_factor,
        "c_baseline_depth": c_baseline_depth.tolist() if isinstance(c_baseline_depth, np.ndarray) else c_baseline_depth,
        "c_mean_reversion_speed": c_mean_reversion_speed.tolist() if isinstance(c_mean_reversion_speed, np.ndarray) else c_mean_reversion_speed,
        "c_self_jump_size": c_self_jump_size.tolist() if isinstance(c_self_jump_size, np.ndarray) else c_self_jump_size,
        "c_mutual_jump_size": c_mutual_jump_size.tolist() if isinstance(c_mutual_jump_size, np.ndarray) else c_mutual_jump_size,
        "c_synchrony_factor": c_synchrony_factor.tolist() if isinstance(c_synchrony_factor, np.ndarray) else c_synchrony_factor,
    },
    "seeds": {
        "train_seed": int(train_seed),
        "network_seed": int(network_seed),
        "eval_seed": int(eval_seed),
        "note": "Only eval_seed affects the evaluation simulation. train_seed and network_seed are included for record-keeping.",
    },
    "evaluation": {
        "n_eval_episodes": int(n_eval_episodes),
        "deterministic_policy": bool(deterministic),
    },
    "files": {
        "histogram_pdf": f"{base_name}_histogram.pdf",
        "histogram_png": f"{base_name}_histogram.png",
        "episode_rewards_txt": os.path.basename(rewards_txt_path),
        "episode_rewards_csv": os.path.basename(rewards_csv_path),
        "summary_txt": os.path.basename(summary_txt_path),
        "summary_json": os.path.basename(summary_json_path),
    },
    "summary_statistics": summary,
}

with open(summary_json_path, "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2)

with open(summary_txt_path, "w", encoding="utf-8") as f:
    f.write(f"run_tag: {meta['run_tag']}\n")
    f.write(f"trial_folder: {meta['trial_folder']}\n")
    f.write(f"assets_tag: {meta['assets_tag']}\n")
    f.write(f"use_best_model: {meta['use_best_model']}\n")
    f.write(f"source_model_path: {meta['source_model_path']}\n")
    f.write(f"figure_dir: {meta['figure_dir']}\n")
    f.write(f"data_dir: {meta['data_dir']}\n")

    f.write("\nENVIRONMENT\n")
    for k, v in meta["environment"].items():
        f.write(f"{k}: {v}\n")

    f.write("\nREWARD SETTINGS\n")
    for k, v in meta["reward_settings"].items():
        f.write(f"{k}: {v}\n")

    f.write("\nDYNAMICS OVERRIDES\n")
    for k, v in meta["dynamics_overrides"].items():
        f.write(f"{k}: {v}\n")

    f.write("\nSEEDS\n")
    for k, v in meta["seeds"].items():
        f.write(f"{k}: {v}\n")

    f.write("\nEVALUATION\n")
    for k, v in meta["evaluation"].items():
        f.write(f"{k}: {v}\n")

    f.write("\nFILES\n")
    for k, v in meta["files"].items():
        f.write(f"{k}: {v}\n")

    f.write("\nSUMMARY STATISTICS\n")
    for k, v in meta["summary_statistics"].items():
        f.write(f"{k}: {v}\n")

# ============================================================
# 14. PLOT + SAVE HISTOGRAM
# ============================================================
hist_pdf_path = os.path.join(fig_dir, f"{base_name}_histogram.pdf")
hist_png_path = os.path.join(fig_dir, f"{base_name}_histogram.png")

plt.figure(figsize=(8, 5))
plt.hist(episode_rewards, bins=60)
plt.xlabel("Terminal cumulative reward", fontsize=11)
plt.ylabel("Frequency", fontsize=11)
plt.title(f"PnL distribution — {run_tag}", fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()

plt.savefig(hist_pdf_path, bbox_inches="tight")
plt.savefig(hist_png_path, dpi=600, bbox_inches="tight")
plt.show()

# ============================================================
# 15. FINAL PRINTS
# ============================================================
print("\nSaved:")
print("  Histogram PDF :", hist_pdf_path)
print("  Histogram PNG :", hist_png_path)
print("  Rewards TXT   :", rewards_txt_path)
print("  Rewards CSV   :", rewards_csv_path)
print("  Summary TXT   :", summary_txt_path)
print("  Summary JSON  :", summary_json_path)

In [ ]:
import os
PROJECT_ROOT = os.environ.get("MBT_PROJECT_ROOT")
if not PROJECT_ROOT:
    PROJECT_ROOT = os.getcwd()
    while not os.path.isdir(os.path.join(PROJECT_ROOT, "mbt_gym")) and os.path.dirname(PROJECT_ROOT) != PROJECT_ROOT:
        PROJECT_ROOT = os.path.dirname(PROJECT_ROOT)

# ============================================================
# MO
# COMBINED TERMINAL PnL DISTRIBUTION (MULTIPLE RHO) 
# Models are trained under different cross-asset-influence terms and simulations are (evaluated) run in 
# each models respective environments (NOT in a single environment) . 
# Above (in the code above) one simulation at a time was run (becuase each checkpoints respective simulation environment is 
# being used) 
# Here just distributions are being plotted together using "Already Saved Data" 
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# GLOBAL STYLE (KEEP CONSISTENT)
# ============================================================
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman", "CMU Serif", "DejaVu Serif"],
    "mathtext.fontset": "cm",
    "axes.labelsize": 26,
    "xtick.labelsize": 20,
    "ytick.labelsize": 20,
    "axes.linewidth": 1.2,
    "lines.linewidth": 6,
})

# ============================================================
# PATHS
# ============================================================
assets_tag = "2Assets"

data_root = os.path.join(PROJECT_ROOT, f"N_figures/{assets_tag}/PnL_Data")
fig_root  = os.path.join(PROJECT_ROOT, f"N_figures/{assets_tag}/PnL_Graph/Combined_Trials")

os.makedirs(fig_root, exist_ok=True)

# ============================================================
# SETTINGS
# ============================================================
rho_list = [1.0, 3.0, 5.0, 7.0, 9.0]

# ============================================================
# UPDATED MAPPING (THIS IS THE ONLY CHANGE)
# ============================================================
rho_map = {
    1.0: ("PPO_2Assets_Trial8",  "PPO_2Assets_Trial8_100M"),
    3.0: ("PPO_2Assets_Trial9",  "PPO_2Assets_Trial9_100M"),
    5.0: ("PPO_2Assets_Trial10", "PPO_2Assets_Trial10_100M"),
    7.0: ("PPO_2Assets_Trial11", "PPO_2Assets_Trial11_100M"),
    9.0: ("PPO_2Assets_Trial12", "PPO_2Assets_Trial12_100M"),
}

# ============================================================
# STORAGE
# ============================================================
all_rewards = []
summary_rows = []

# ============================================================
# LOAD DATA
# ============================================================
for rho in rho_list:

    trial_folder, run_tag = rho_map[rho]

    file_path = os.path.join(
        data_root,
        trial_folder,
        f"{run_tag}_PnL_episode_rewards.csv"
    )

    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Missing file: {file_path}")

    rewards = np.loadtxt(file_path, delimiter=",", skiprows=1)

    n = len(rewards)
    mean = np.mean(rewards)
    var = np.var(rewards, ddof=1)
    std = np.sqrt(var)
    sharpe = mean / std if std > 0 else np.nan

    all_rewards.append(rewards)

    summary_rows.append({
        "rho": rho,
        "mean": mean,
        "variance": var,
        "std": std,
        "sharpe": sharpe,
        "n": n
    })

    print(f"Loaded rho={rho:.1f} | n={n}")

# ============================================================
# SAVE SUMMARY
# ============================================================
summary_df = pd.DataFrame(summary_rows)

csv_path = os.path.join(fig_root, "PnL_summary_by_rho.csv")
json_path = os.path.join(fig_root, "PnL_summary_by_rho.json")

summary_df.to_csv(csv_path, index=False)
summary_df.to_json(json_path, orient="records", indent=4)

print("\nSaved summary:")
print(csv_path)
print(json_path)

# ============================================================
# GLOBAL BINS (NO TRIMMING)
# ============================================================
global_min = min(np.min(x) for x in all_rewards)
global_max = max(np.max(x) for x in all_rewards)

bins = np.linspace(global_min, global_max, 81)

# ============================================================
# HISTOGRAM OVERLAY
# ============================================================
plt.figure(figsize=(10,6))

for rewards, rho in zip(all_rewards, rho_list):
    plt.hist(
        rewards,
        bins=bins,
        alpha=0.35,
        density=True,
        label=f"ρ={rho:.1f}"
    )

plt.xlabel("Terminal PnL")
#plt.ylabel("Relative Frequency")
plt.ylabel("Density")
plt.legend(fontsize=20)
plt.grid(True, linestyle="--", linewidth=0.8, alpha=0.7)
plt.tick_params(width=1.5, length=6)

plt.tight_layout()

overlay_pdf = os.path.join(fig_root, "PnL_histogram_by_rho_overlay.pdf")
overlay_png = os.path.join(fig_root, "PnL_histogram_by_rho_overlay.png")

plt.savefig(overlay_pdf, bbox_inches="tight")
plt.savefig(overlay_png, dpi=600, bbox_inches="tight")

print("\nSaved overlay histogram:")
print(overlay_pdf)
print(overlay_png)

plt.show()

# ============================================================
# LINE HISTOGRAM (SMOOTH)
# ============================================================
plt.figure(figsize=(10,6))

for rewards, rho in zip(all_rewards, rho_list):
    counts, edges = np.histogram(rewards, bins=bins, density=True)
    centers = 0.5 * (edges[:-1] + edges[1:])
    plt.plot(centers, counts, label=f"ρ={rho:.1f}")

plt.xlabel("Terminal PnL")
#plt.ylabel("Relative Frequency")
plt.ylabel("Density")
plt.legend(fontsize=20)
plt.grid(True, linestyle="--", linewidth=0.8, alpha=0.7)
plt.tick_params(width=1.5, length=6)

plt.tight_layout()

lines_pdf = os.path.join(fig_root, "PnL_histogram_by_rho_lines.pdf")
lines_png = os.path.join(fig_root, "PnL_histogram_by_rho_lines.png")

plt.savefig(lines_pdf, bbox_inches="tight")
plt.savefig(lines_png, dpi=600, bbox_inches="tight")

print("\nSaved line histogram:")
print(lines_pdf)
print(lines_png)

plt.show()

In [ ]:
# MO
# Checking if means and Variances of Terminal PnL (computed in the first code block) 
# are different statistically using Welch and F Tests. 
# Not simulating. Just using already simulated data earlier.
# Models are trained under different cross-asset-influence terms and simulations are run in 
# each models respective environments (NOT in a single environment) 

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# ============================================================
# GLOBAL STYLE (LaTeX-like CMR)
# ============================================================
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman", "CMU Serif", "DejaVu Serif"],
    "mathtext.fontset": "cm",
    "axes.labelsize": 26,
    "xtick.labelsize": 20,
    "ytick.labelsize": 20,
    "axes.linewidth": 1.2,
    "lines.linewidth": 6,
})

# ============================================================
# 1. SETTINGS
# ============================================================
assets_tag = "2Assets"

# --- portable project root: auto-detect the folder that contains "N_figures"
#     (works on both the Windows lab machine and the Mac); falls back to CWD ---
_here = os.getcwd()
_p = _here
while _p != os.path.dirname(_p):
    if os.path.isdir(os.path.join(_p, "N_figures")):
        PROJECT_ROOT = _p
        break
    _p = os.path.dirname(_p)
else:
    PROJECT_ROOT = _here

base_data_root = os.path.join(PROJECT_ROOT, "N_figures", assets_tag, "PnL_Data")
base_fig_root  = os.path.join(PROJECT_ROOT, "N_figures", assets_tag, "PnL_Graph", "Combined_Trials2")

os.makedirs(base_fig_root, exist_ok=True)

# ============================================================
# 2. MAP PARAMETER -> FILE
# ============================================================
rho_map = {
    1.0: ("PPO_2Assets_Trial8",  "PPO_2Assets_Trial8_100M"),
    3.0: ("PPO_2Assets_Trial9",  "PPO_2Assets_Trial9_100M"),
    5.0: ("PPO_2Assets_Trial10", "PPO_2Assets_Trial10_100M"),
    7.0: ("PPO_2Assets_Trial11", "PPO_2Assets_Trial11_100M"),
    9.0: ("PPO_2Assets_Trial12", "PPO_2Assets_Trial12_100M"),
}

param_values = sorted(rho_map.keys())

csv_paths = []
labels = []

for param in param_values:
    trial_folder, run_tag = rho_map[param]
    csv_path = os.path.join(
        base_data_root,
        trial_folder,
        f"{run_tag}_PnL_episode_rewards.csv"
    )
    csv_paths.append(csv_path)
    labels.append(float(param))

print("Files to load:")
for p in csv_paths:
    print(" ", p)

# ============================================================
# 3. LOAD DATA + SUMMARY STATS
# ============================================================
all_rewards = []
summary_rows = []

print("\n===== SUMMARY BY PARAMETER =====\n")

for csv_path, param in zip(csv_paths, labels):

    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"Missing file: {csv_path}")

    df = pd.read_csv(csv_path)

    if "terminal_cumulative_reward" not in df.columns:
        raise KeyError(
            f"'terminal_cumulative_reward' column not found in file:\n{csv_path}\n"
            f"Available columns: {list(df.columns)}"
        )

    rewards = df["terminal_cumulative_reward"].dropna().to_numpy(dtype=float)

    if len(rewards) < 2:
        raise ValueError(f"Not enough observations in file: {csv_path}")

    all_rewards.append(rewards)

    n_val = len(rewards)
    mean_val = np.mean(rewards)
    var_val = np.var(rewards, ddof=1)
    std_val = np.sqrt(var_val)
    sem_val = std_val / np.sqrt(n_val)
    sharpe_val = mean_val / std_val if std_val > 0 else np.nan

    ci95_low = mean_val - 1.96 * sem_val
    ci95_high = mean_val + 1.96 * sem_val

    summary_rows.append({
        "parameter": float(param),
        "n": int(n_val),
        "mean": float(mean_val),
        "variance": float(var_val),
        "std": float(std_val),
        "sem": float(sem_val),
        "ci95_low": float(ci95_low),
        "ci95_high": float(ci95_high),
        "sharpe": float(sharpe_val) if not np.isnan(sharpe_val) else None,
    })

    print(
        f"param = {param:.2f}   "
        f"n = {n_val:d}   "
        f"mean = {mean_val:.8f}   "
        f"variance = {var_val:.8f}   "
        f"std = {std_val:.8f}   "
        f"SEM = {sem_val:.10f}   "
        f"CI95 = [{ci95_low:.8f}, {ci95_high:.8f}]   "
        f"Sharpe = {sharpe_val:.8f}"
    )

summary_df = pd.DataFrame(summary_rows)

summary_csv = os.path.join(base_fig_root, "PnL_summary_by_parameter.csv")
summary_json = os.path.join(base_fig_root, "PnL_summary_by_parameter.json")

summary_df.to_csv(summary_csv, index=False)
summary_df.to_json(summary_json, orient="records", indent=4)

print("\nSaved summary table:")
print(" ", summary_csv)
print(" ", summary_json)

# ============================================================
# 4. PAIRWISE WELCH T-TESTS
# ============================================================
def welch_df(x, y):
    n1, n2 = len(x), len(y)
    s1, s2 = np.var(x, ddof=1), np.var(y, ddof=1)
    return ((s1/n1 + s2/n2)**2) / (((s1/n1)**2)/(n1-1) + ((s2/n2)**2)/(n2-1))

pairwise_rows = []

print("\n===== PAIRWISE WELCH TESTS =====\n")

for i in range(len(all_rewards) - 1):
    x = all_rewards[i]
    y = all_rewards[i + 1]

    t_stat, p_two = stats.ttest_ind(x, y, equal_var=False)
    df_w = welch_df(x, y)

    # 95% Welch CI for the difference in means (mu_x - mu_y), using the Welch df above
    n1, n2 = len(x), len(y)
    mean_diff = np.mean(x) - np.mean(y)
    se_diff = np.sqrt(np.var(x, ddof=1) / n1 + np.var(y, ddof=1) / n2)
    t_crit = stats.t.ppf(0.975, df_w)
    mean_diff_ci_low = mean_diff - t_crit * se_diff
    mean_diff_ci_high = mean_diff + t_crit * se_diff

    print(f"{labels[i]:.2f} vs {labels[i+1]:.2f}")
    print(f"t = {t_stat:.6f}")
    print(f"Welch df = {df_w:.6f}")
    print(f"p = {p_two:.12g}")
    print(f"mean diff = {mean_diff:.8f}   SE = {se_diff:.10f}   "
          f"95% CI = [{mean_diff_ci_low:.8f}, {mean_diff_ci_high:.8f}]")
    print()

    pairwise_rows.append({
        "comparison": f"{labels[i]:.2f} vs {labels[i+1]:.2f}",
        "t_stat": float(t_stat),
        "welch_df": float(df_w),
        "p_value": float(p_two),
        "mean_diff": float(mean_diff),
        "se_diff": float(se_diff),
        "mean_diff_ci95_low": float(mean_diff_ci_low),
        "mean_diff_ci95_high": float(mean_diff_ci_high),
    })

pairwise_df = pd.DataFrame(pairwise_rows)

pairwise_csv = os.path.join(base_fig_root, "PnL_pairwise_welch_tests.csv")
pairwise_json = os.path.join(base_fig_root, "PnL_pairwise_welch_tests.json")

pairwise_df.to_csv(pairwise_csv, index=False)
pairwise_df.to_json(pairwise_json, orient="records", indent=4)

print("Saved pairwise results:")
print(" ", pairwise_csv)
print(" ", pairwise_json)

# ============================================================
# 5. PAIRWISE F-TESTS
# ============================================================
# Bootstrap settings for the (normality-free) variance-ratio CI
BOOT_B = 2000        # number of bootstrap resamples (raise for tighter percentile estimates)
BOOT_SEED = 12345    # for reproducibility
boot_rng = np.random.default_rng(BOOT_SEED)

f_rows = []

print("\n===== PAIRWISE F-TESTS =====\n")

for i in range(len(all_rewards) - 1):
    var_x = np.var(all_rewards[i], ddof=1)
    var_y = np.var(all_rewards[i+1], ddof=1)

    f_stat = var_x / var_y

    # ---- 95% CIs for the variance ratio sigma_x^2 / sigma_y^2 ----
    x = all_rewards[i]
    y = all_rewards[i + 1]
    n1, n2 = len(x), len(y)
    d1, d2 = n1 - 1, n2 - 1

    # (a) F-based CI (parametric; assumes normality)
    f_ci_low = f_stat / stats.f.ppf(0.975, d1, d2)
    f_ci_high = f_stat / stats.f.ppf(0.025, d1, d2)

    # (b) Bootstrap percentile CI (no normality assumption)
    boot_ratios = np.empty(BOOT_B, dtype=float)
    for b in range(BOOT_B):
        xb = x[boot_rng.integers(0, n1, size=n1)]
        yb = y[boot_rng.integers(0, n2, size=n2)]
        boot_ratios[b] = np.var(xb, ddof=1) / np.var(yb, ddof=1)
    boot_ci_low, boot_ci_high = np.percentile(boot_ratios, [2.5, 97.5])

    print(f"{labels[i]:.2f} vs {labels[i+1]:.2f}")
    print(f"F = {f_stat:.6f}")
    print(f"F-based 95% CI (var ratio)    = [{f_ci_low:.6f}, {f_ci_high:.6f}]")
    print(f"Bootstrap 95% CI (var ratio)  = [{boot_ci_low:.6f}, {boot_ci_high:.6f}]")
    print()

    f_rows.append({
        "comparison": f"{labels[i]:.2f} vs {labels[i+1]:.2f}",
        "f_stat": float(f_stat),
        "f_ci95_low": float(f_ci_low),
        "f_ci95_high": float(f_ci_high),
        "boot_ci95_low": float(boot_ci_low),
        "boot_ci95_high": float(boot_ci_high),
    })

f_df_out = pd.DataFrame(f_rows)

f_csv = os.path.join(base_fig_root, "PnL_pairwise_f_tests.csv")
f_json = os.path.join(base_fig_root, "PnL_pairwise_f_tests.json")

f_df_out.to_csv(f_csv, index=False)
f_df_out.to_json(f_json, orient="records", indent=4)

print("Saved F-test results:")
print(" ", f_csv)
print(" ", f_json)

# ============================================================
# 6–8 PLOTS
# ============================================================

params = summary_df["parameter"].to_numpy()
means = summary_df["mean"].to_numpy()
stds  = summary_df["std"].to_numpy()

# --- MEAN ---
plt.figure(figsize=(8.5,5.5))
plt.plot(params, means, "o-", linewidth=3.5)
plt.xlabel(r"Cross-asset influence parameter ($\rho$)")
plt.ylabel("Mean Terminal PnL")
plt.xticks([1,3,5,7,9])
plt.grid(True, linestyle="--", linewidth=0.8, alpha=0.7)
plt.tick_params(width=1.5, length=6)
plt.tight_layout()

pdf_path = os.path.join(base_fig_root,"PnL_mean_with_95CI.pdf")
png_path = os.path.join(base_fig_root,"PnL_mean_with_95CI.png")

plt.savefig(pdf_path, bbox_inches="tight")
plt.savefig(png_path, dpi=600, bbox_inches="tight")

print("\nSaved mean plot:")
print(" ", pdf_path)
print(" ", png_path)

plt.show()

# --- STD ---
plt.figure(figsize=(8.5,5.5))
plt.plot(params, stds, "o-", linewidth=3.5)
plt.xlabel(r"Cross-asset influence parameter ($\rho$)")
plt.ylabel("Standard deviation of terminal PnL")
plt.xticks([1,3,5,7,9])
plt.grid(True, linestyle="--", linewidth=0.8, alpha=0.7)
plt.tick_params(width=1.5, length=6)
plt.tight_layout()

pdf_path = os.path.join(base_fig_root,"PnL_sd_over_rho.pdf")
png_path = os.path.join(base_fig_root,"PnL_sd_over_rho.png")

plt.savefig(pdf_path, bbox_inches="tight")
plt.savefig(png_path, dpi=600, bbox_inches="tight")

print("\nSaved std plot:")
print(" ", pdf_path)
print(" ", png_path)

plt.show()

# ============================================================
# 8.5 COMBINED MEAN + STD (NEW)
# ============================================================
#fig, ax1 = plt.subplots(figsize=(8.5, 5.5))
fig, ax1 = plt.subplots(figsize=(9, 5.5))

ax1.plot(params, means, "o-", color="blue", label="Mean", ms=15, linewidth=6)
ax1.set_xlabel(r"Cross-asset influence ($\rho$)")
ax1.set_ylabel("Mean")

ax1.set_xticks([1,3,5,7,9])
ax1.tick_params(width=1.5, length=6)

ax1.grid(True, linestyle="--", linewidth=0.8, alpha=0.7)

ax2 = ax1.twinx()
ax2.plot(params, stds, "s--", color="red", label="SD", ms=15, linewidth=6)
ax2.set_ylabel("Standard Deviation")
ax2.tick_params(width=1.5, length=6)

lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
ax1.legend(lines_1 + lines_2, labels_1 + labels_2, fontsize=16)
# ax1.legend(lines_1 + lines_2, labels_1 + labels_2, fontsize=12)

plt.tight_layout()

combined_pdf = os.path.join(base_fig_root, "PnL_mean_sd_combined.pdf")
combined_png = os.path.join(base_fig_root, "PnL_mean_sd_combined.png")

plt.savefig(combined_pdf, bbox_inches="tight")
plt.savefig(combined_png, dpi=600, bbox_inches="tight")

print("\nSaved combined mean + std plot:")
print(" ", combined_pdf)
print(" ", combined_png)

plt.show()

# --- SHARPE ---
#plt.figure(figsize=(8.5,5.5))
plt.figure(figsize=(9,5.5))
#plt.plot(params, summary_df["sharpe"], "o-", linewidth=3.5)
plt.plot(params, summary_df["sharpe"], "o-", linewidth=6, color="green", ms=15)
plt.xlabel(r"Cross-asset influence ($\rho$)")
#plt.xlabel(r"Cross-asset influence parameter ($\rho$)")
plt.ylabel("Sharpe Ratio")
plt.xticks([1,3,5,7,9])
plt.grid(True, linestyle="--", linewidth=0.8, alpha=0.7)
plt.tick_params(width=1.5, length=6)
plt.tight_layout()

pdf_path = os.path.join(base_fig_root,"PnL_sharpe_over_rho.pdf")
png_path = os.path.join(base_fig_root,"PnL_sharpe_over_rho.png")

plt.savefig(pdf_path, bbox_inches="tight")
plt.savefig(png_path, dpi=600, bbox_inches="tight")

print("\nSaved sharpe plot:")
print(" ", pdf_path)
print(" ", png_path)

plt.show()

In [ ]:
# ====================================
# MO >>> COST of IGNORANCE : What happens when rho exists but MM ignores it
# Being evaluated (simulated) here : DO NOT rerun (takes time and updates the saved data)
# FIXED MODEL → MULTIPLE ENVIRONMENTS
# One Fixed Model trained once (rho=0  and rho_c=0), evaluated under different MO cross-asset influence
# terms rho={1,3,5,7,9}
# ============================================================

import os
import sys
import json
import numpy as np
import matplotlib.pyplot as plt

from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import VecMonitor

# ============================================================
# 0. PROJECT PATH
# ============================================================
import os
PROJECT_ROOT = os.environ.get("MBT_PROJECT_ROOT")
if not PROJECT_ROOT:
    PROJECT_ROOT = os.getcwd()
    while not os.path.isdir(os.path.join(PROJECT_ROOT, "mbt_gym")) and os.path.dirname(PROJECT_ROOT) != PROJECT_ROOT:
        PROJECT_ROOT = os.path.dirname(PROJECT_ROOT)

if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

import mbt_gym  # noqa

# ============================================================
# 1. IMPORT YOUR MODULES
# ============================================================
from environment_set_up.N_env_setup import make_sb_env, make_eval_sb_env
from environment_set_up.N_env_builders import get_cj_env, set_env_globals

# ============================================================
# 2. FIXED MODEL (TRAINED ONCE)
# ============================================================
trial_folder = "PPO_2Assets_Trial0"
run_tag = "PPO_2Assets_Trial0_100M"

checkpoint_root = os.path.join(PROJECT_ROOT, "N_SB_models/PPO_Checkpoints_2Assets")

def build_checkpoint_path(root, trial_folder, run_tag):
    return os.path.join(root, trial_folder, f"{run_tag}.zip")

model_path = build_checkpoint_path(checkpoint_root, trial_folder, run_tag)

if not os.path.exists(model_path):
    raise FileNotFoundError(f"Model not found: {model_path}")

print(f"Using model: {model_path}")

# ============================================================
# 3. EVALUATION ENVIRONMENTS (ρ VALUES)
# ============================================================
rho_eval_list = [1.0, 3.0, 5.0, 7.0, 9.0]

# ============================================================
# 4. SETTINGS
# ============================================================
terminal_time = 1.0
n_steps = 100
num_trajectories = 1000

train_seed = 123
eval_seed = 456

eval_phi = 0.0
eval_alpha = 0.2

n_eval_episodes = 100000
deterministic = True

# ============================================================
# 5. OTHER PARAMS (UNCHANGED)
# ============================================================
baseline_arrival_rate = None
mean_reversion_speed = None
self_jump_size = None
mutual_jump_size = None
synchrony_factor = None

c_baseline_depth = None
c_mean_reversion_speed = None
c_self_jump_size = None
c_mutual_jump_size = None
c_synchrony_factor = None

sigma = None
initial_price = None
initial_inventory = None

# ============================================================
# 6. SAVE DIR
# ============================================================
assets_tag = "2Assets"

def build_save_dirs(rho):
    fig_root = os.path.join(PROJECT_ROOT, f"N_figures/{assets_tag}/PnL_Graph_FixedModelEval")
    data_root = os.path.join(PROJECT_ROOT, f"N_figures/{assets_tag}/PnL_Data_FixedModelEval")

    fig_dir = os.path.join(fig_root, f"rho_{rho}")
    data_dir = os.path.join(data_root, f"rho_{rho}")

    os.makedirs(fig_dir, exist_ok=True)
    os.makedirs(data_dir, exist_ok=True)

    return fig_dir, data_dir

# ============================================================
# 7. LOAD MODEL
# ============================================================
def load_model_robust(path, env):
    try:
        return PPO.load(path, env=env, device="cpu")
    except Exception:
        return PPO.load(
            path,
            env=env,
            device="cpu",
            custom_objects={
                "learning_rate": 0.0,
                "lr_schedule": lambda _: 0.0,
                "clip_range": lambda _: 0.2,
            },
        )

# ============================================================
# 8. EVAL FUNCTION
# ============================================================
def collect_rewards(model, vec_env, n_episodes):
    rewards = []
    obs = vec_env.reset()

    while len(rewards) < n_episodes:
        actions, _ = model.predict(obs, deterministic=True)
        obs, _, dones, infos = vec_env.step(actions)

        for done, info in zip(dones, infos):
            if done:
                rewards.append(float(info["episode"]["r"]))

    return np.array(rewards[:n_episodes])

def summarize(x):
    return {
        "mean": float(np.mean(x)),
        "std": float(np.std(x, ddof=1)),
        "q05": float(np.quantile(x, 0.05)),
        "q50": float(np.quantile(x, 0.50)),
        "q95": float(np.quantile(x, 0.95)),
    }

# ============================================================
# 9. MAIN LOOP (OVER ENVIRONMENTS)
# ============================================================
all_results = {}

for rho in rho_eval_list:

    print("\n" + "="*60)
    print(f"Evaluating environment with cross_asset_influence = {rho}")
    print("="*60)

    # --------------------------------------------------------
    # Build matrices
    # --------------------------------------------------------
    eval_cross = np.array([[0.0, rho], [rho, 0.0]])
    eval_c_cross = np.zeros((2, 2))

    fig_dir, data_dir = build_save_dirs(rho)

    # --------------------------------------------------------
    # ENV KWARGS
    # --------------------------------------------------------
    env_kwargs = dict(
        cross_asset_influence=eval_cross,
        c_cross_asset_influence=eval_c_cross,
    )

    # --------------------------------------------------------
    # BUILD ENV
    # --------------------------------------------------------
    set_env_globals(
        terminal_time=terminal_time,
        n_steps=n_steps,
        phi=eval_phi,
        alpha=eval_alpha,
        train_seed=train_seed,
    )

    train_bundle = make_sb_env(
        get_env_fn=get_cj_env,
        num_trajectories=num_trajectories,
        num_assets=2,
        initial_inventory=initial_inventory,
        do_reward_scaling=False,
        env_seed=train_seed,
        **env_kwargs,
    )

    eval_bundle = make_eval_sb_env(
        get_env_fn=get_cj_env,
        train_bundle=train_bundle,
        eval_seed=eval_seed,
        **env_kwargs,
    )

    sb_env = VecMonitor(eval_bundle.sb_eval_env)

    # --------------------------------------------------------
    # LOAD MODEL
    # --------------------------------------------------------
    model = load_model_robust(model_path, sb_env)

    # --------------------------------------------------------
    # RUN EVAL
    # --------------------------------------------------------
    rewards = collect_rewards(model, sb_env, n_eval_episodes)
    summary = summarize(rewards)

    print("Mean:", summary["mean"], "Std:", summary["std"])

    # --------------------------------------------------------
    # SAVE DATA
    # --------------------------------------------------------
    base_name = f"{run_tag}_rho{rho}"

    np.savetxt(os.path.join(data_dir, f"{base_name}.csv"), rewards)

    with open(os.path.join(data_dir, f"{base_name}_summary.json"), "w") as f:
        json.dump(summary, f, indent=2)

    # --------------------------------------------------------
    # HISTOGRAM
    # --------------------------------------------------------
    plt.figure(figsize=(8,5))
    plt.hist(rewards, bins=60)
    plt.title(f"PnL — Fixed Model | rho={rho}")
    plt.grid(True)
    plt.tight_layout()

    plt.savefig(os.path.join(fig_dir, f"{base_name}.pdf"))
    plt.close()

    all_results[rho] = summary

# ============================================================
# 10. COMBINED SUMMARY
# ============================================================
combined_path = os.path.join(PROJECT_ROOT, f"N_figures/{assets_tag}/PnL_Data_FixedModelEval/combined.json")

with open(combined_path, "w") as f:
    json.dump(all_results, f, indent=2)

print("\nDONE. Combined summary saved.")

In [ ]:
import os
PROJECT_ROOT = os.environ.get("MBT_PROJECT_ROOT")
if not PROJECT_ROOT:
    PROJECT_ROOT = os.getcwd()
    while not os.path.isdir(os.path.join(PROJECT_ROOT, "mbt_gym")) and os.path.dirname(PROJECT_ROOT) != PROJECT_ROOT:
        PROJECT_ROOT = os.path.dirname(PROJECT_ROOT)

# MO
# COST of IGNORANCE : Combined Results
# Data is loaded. Data comes from simulations previously (in the code above) done
# one fixed base model (rho=0  and rho_c=0) in training and rho is varied in simulation rho={1,3,5,7,9}

# Combined Results:
# PnL Distributions plotted together
# mean pnl, sd and sharp ratios are plotted over rho
# Welch and F tests used to see if means and variances are differnet (Check the hyposthesis) 

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# ============================================================
# GLOBAL STYLE (LaTeX-like CMR)
# ============================================================
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman", "CMU Serif", "DejaVu Serif"],
    "mathtext.fontset": "cm",
    "axes.labelsize": 26,
    "xtick.labelsize": 20,
    "ytick.labelsize": 20,
    "axes.linewidth": 1.2,
    "lines.linewidth": 6,
})

# ============================================================
# PATH
# ============================================================
assets_tag = "2Assets"

data_root = os.path.join(PROJECT_ROOT, f"N_figures/{assets_tag}/PnL_Data_FixedModelEval")
fig_root  = os.path.join(PROJECT_ROOT, f"N_figures/{assets_tag}/PnL_Graph_FixedModelEval")

os.makedirs(fig_root, exist_ok=True)

rho_list = [1.0, 3.0, 5.0, 7.0, 9.0]
run_tag = "PPO_2Assets_Trial0_100M"

# ============================================================
# STORAGE
# ============================================================
all_rewards = []
means = []
stds = []
sharpes = []
summary_rows = []

print("\n===== SUMMARY BY PARAMETER =====\n")

# ============================================================
# LOAD + COMPUTE
# ============================================================
for rho in rho_list:
    data_dir = os.path.join(data_root, f"rho_{rho}")
    file_path = os.path.join(data_dir, f"{run_tag}_rho{rho}.csv")

    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Missing file: {file_path}")

    rewards = np.loadtxt(file_path)

    n = len(rewards)
    mean = np.mean(rewards)
    var = np.var(rewards, ddof=1)
    std = np.sqrt(var)
    sem = std / np.sqrt(n)
    sharpe = mean / std if std > 0 else np.nan

    ci_low = mean - 1.96 * sem
    ci_high = mean + 1.96 * sem

    all_rewards.append(rewards)
    means.append(mean)
    stds.append(std)
    sharpes.append(sharpe)

    summary_rows.append({
        "rho": rho,
        "n": n,
        "mean": mean,
        "variance": var,
        "std": std,
        "sem": sem,
        "ci95_low": ci_low,
        "ci95_high": ci_high,
        "sharpe": sharpe if not np.isnan(sharpe) else None
    })

    print(
        f"rho = {rho:.2f}   "
        f"n = {n:d}   "
        f"mean = {mean:.8f}   "
        f"variance = {var:.8f}   "
        f"std = {std:.8f}   "
        f"SEM = {sem:.10f}   "
        f"CI95 = [{ci_low:.8f}, {ci_high:.8f}]   "
        f"Sharpe = {sharpe:.8f}"
    )

summary_df = pd.DataFrame(summary_rows)

summary_csv = os.path.join(fig_root, "PnL_summary_fixed_model.csv")
summary_json = os.path.join(fig_root, "PnL_summary_fixed_model.json")

summary_df.to_csv(summary_csv, index=False)
summary_df.to_json(summary_json, orient="records", indent=4)

print("\nSaved summary table:")
print(" ", summary_csv)
print(" ", summary_json)

# ============================================================
# TEST FUNCTIONS (UNCHANGED)
# ============================================================

def welch_df(x, y):
    n1, n2 = len(x), len(y)
    s1, s2 = np.var(x, ddof=1), np.var(y, ddof=1)
    return ((s1/n1 + s2/n2)**2) / (((s1/n1)**2)/(n1-1) + ((s2/n2)**2)/(n2-1))


def one_sided_welch(x, y):
    t_stat, p_two = stats.ttest_ind(x, y, equal_var=False)

    if t_stat >= 0:
        p_one = p_two / 2.0
    else:
        p_one = 1.0 - p_two / 2.0

    return t_stat, p_two, p_one


def one_sided_f_test(x, y):
    var_x = np.var(x, ddof=1)
    var_y = np.var(y, ddof=1)

    f_stat = var_x / var_y
    df1, df2 = len(x)-1, len(y)-1

    p_two = 2 * min(
        stats.f.cdf(f_stat, df1, df2),
        1 - stats.f.cdf(f_stat, df1, df2)
    )

    if f_stat <= 1:
        p_one = stats.f.cdf(f_stat, df1, df2)
    else:
        p_one = 1 - stats.f.cdf(f_stat, df1, df2)

    return f_stat, p_two, p_one, df1, df2


# ============================================================
# STATISTICAL TESTS (ADDED BACK — THIS WAS MISSING)
# ============================================================

print("\n===== STATISTICAL TESTS (vs rho=1.0 baseline) =====\n")

base = all_rewards[0]

for i in range(1, len(all_rewards)):
    comp = all_rewards[i]
    rho_val = rho_list[i]

    t_stat, p_two, p_one = one_sided_welch(comp, base)
    df_val = welch_df(comp, base)

    f_stat, f_p_two, f_p_one, df1, df2 = one_sided_f_test(comp, base)

    print(f"rho = {rho_val:.1f} vs rho = 1.0")
    print(f"  Welch t-stat = {t_stat:.6f}, df ≈ {df_val:.2f}")
    print(f"  p-value (two-sided) = {p_two:.6e}, p-value (one-sided) = {p_one:.6e}")
    print(f"  F-stat = {f_stat:.6f}, df1={df1}, df2={df2}")
    print(f"  p-value F (two-sided) = {f_p_two:.6e}, p-value F (one-sided) = {f_p_one:.6e}")
    print("")


# ============================================================
# PLOTTING
# ============================================================
rho_arr = np.array(rho_list)
means = np.array(means)
stds = np.array(stds)
sharpes = np.array(sharpes)

# NO TRIMMING
global_min = min(np.min(x) for x in all_rewards)
global_max = max(np.max(x) for x in all_rewards)

bins = np.linspace(global_min, global_max, 81)

# HISTOGRAM OVERLAY
plt.figure(figsize=(10,6))
for rewards, rho in zip(all_rewards, rho_list):
    plt.hist(rewards, bins=bins, alpha=0.35, density=True,
             label=f"ρ={rho:.1f}")

plt.xlabel("Terminal PnL")
plt.ylabel("Density")
plt.legend(fontsize=16)
plt.grid(True, linestyle="--", linewidth=0.8, alpha=0.7)
plt.tick_params(width=1.5, length=6)

plt.xlim(-15, 15)

plt.tight_layout()

path_pdf = os.path.join(fig_root, "PnL_fixed_model_hist_overlay.pdf")
path_png = os.path.join(fig_root, "PnL_fixed_model_hist_overlay.png")

plt.savefig(path_pdf, bbox_inches="tight")
plt.savefig(path_png, dpi=600, bbox_inches="tight")

print("\nSaved histogram overlay:")
print(" ", path_pdf)
print(" ", path_png)

plt.show()

# LINE HIST
plt.figure(figsize=(10,6))
for rewards, rho in zip(all_rewards, rho_list):
    counts, edges = np.histogram(rewards, bins=bins, density=True)
    centers = 0.5*(edges[:-1]+edges[1:])
    plt.plot(centers, counts, label=f"ρ={rho}")

plt.xlabel("Terminal PnL")
plt.ylabel("Density")
plt.legend(fontsize=12)
plt.grid(True, linestyle="--", linewidth=0.8, alpha=0.7)
plt.tick_params(width=1.5, length=6)

plt.xlim(global_min, global_max)

plt.tight_layout()

path_pdf = os.path.join(fig_root, "PnL_fixed_model_hist_lines.pdf")
path_png = os.path.join(fig_root, "PnL_fixed_model_hist_lines.png")

plt.savefig(path_pdf, bbox_inches="tight")
plt.savefig(path_png, dpi=600, bbox_inches="tight")

print("\nSaved line histogram:")
print(" ", path_pdf)
print(" ", path_png)

plt.show()

# MEAN
plt.figure(figsize=(8.5,5.5))
plt.plot(rho_arr, means, "o-")
plt.xlabel(r"Cross-asset influence parameter ($\rho$)")
plt.ylabel("Mean Terminal PnL")
plt.grid(True)
plt.tight_layout()

path_pdf = os.path.join(fig_root, "mean_vs_rho.pdf")
path_png = os.path.join(fig_root, "mean_vs_rho.png")

plt.savefig(path_pdf, bbox_inches="tight")
plt.savefig(path_png, dpi=600, bbox_inches="tight")

print("\nSaved mean plot:")
print(" ", path_pdf)
print(" ", path_png)

plt.show()

# STD
plt.figure(figsize=(8.5,5.5))
plt.plot(rho_arr, stds, "o-")
plt.xlabel(r"Cross-asset influence parameter ($\rho$)")
plt.ylabel("Standard deviation of Terminal PnL")
plt.grid(True)
plt.tight_layout()

path_pdf = os.path.join(fig_root, "std_vs_rho.pdf")
path_png = os.path.join(fig_root, "std_vs_rho.png")

plt.savefig(path_pdf, bbox_inches="tight")
plt.savefig(path_png, dpi=600, bbox_inches="tight")

print("\nSaved std plot:")
print(" ", path_pdf)
print(" ", path_png)

plt.show()

# ============================================================
# NEW: COMBINED MEAN + STD
# ============================================================
#fig, ax1 = plt.subplots(figsize=(8.5, 5.5))
fig, ax1 = plt.subplots(figsize=(9, 5.5))

ax1.plot(rho_arr, means, "o-", color="blue", label="Mean")
ax1.set_xlabel(r"Cross-asset influence ($\rho$)")
ax1.set_ylabel("Mean")
ax1.tick_params(width=1.5, length=6)
ax1.grid(True, linestyle="--", linewidth=0.8, alpha=0.7)

ax2 = ax1.twinx()
ax2.plot(rho_arr, stds, "s--", color="red", label="SD")
ax2.set_ylabel("Standard Deviation")
ax2.tick_params(width=1.5, length=6)

lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
ax1.legend(lines_1 + lines_2, labels_1 + labels_2, fontsize=15, loc="upper left", bbox_to_anchor=(0, 0.85))

plt.tight_layout()

path_pdf = os.path.join(fig_root, "mean_std_combined.pdf")
path_png = os.path.join(fig_root, "mean_std_combined.png")

plt.savefig(path_pdf, bbox_inches="tight")
plt.savefig(path_png, dpi=600, bbox_inches="tight")

print("\nSaved combined mean + std plot:")
print(" ", path_pdf)
print(" ", path_png)

plt.show()

# SHARPE
plt.figure(figsize=(9,5.5))
plt.plot(rho_arr, sharpes, "o-", color="green")
plt.xlabel(r"Cross-asset influence parameter ($\rho$)")
plt.ylabel("Sharp ratio")
plt.grid(True)
plt.tight_layout()

path_pdf = os.path.join(fig_root, "sharpe_vs_rho.pdf")
path_png = os.path.join(fig_root, "sharpe_vs_rho.png")

plt.savefig(path_pdf, bbox_inches="tight")
plt.savefig(path_png, dpi=600, bbox_inches="tight")

print("\nSaved sharpe plot:")
print(" ", path_pdf)
print(" ", path_png)

plt.show()

LOB Simulations 

In [ ]:
# ====================================
# LOB 
# Note: DO NOT Rerun (running evaluations). Takes time and resaves the data files
# Multiple MODELs → MULTIPLE ENVIRONMENTS
# Models are trained (previously) with different values of rho_c={1,2,3,4,5} while rho=0
# And evaluated (Terminal PnL is simulated) in their Respective Environments in the code. 
# Saving Data (samples), Histograms for each sample and summary
# ============================================================

import os
import sys
import json
import numpy as np
import matplotlib.pyplot as plt

from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import VecMonitor

# ============================================================
# 0. PROJECT PATH
# ============================================================
import os
PROJECT_ROOT = os.environ.get("MBT_PROJECT_ROOT")
if not PROJECT_ROOT:
    PROJECT_ROOT = os.getcwd()
    while not os.path.isdir(os.path.join(PROJECT_ROOT, "mbt_gym")) and os.path.dirname(PROJECT_ROOT) != PROJECT_ROOT:
        PROJECT_ROOT = os.path.dirname(PROJECT_ROOT)

if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

import mbt_gym  # noqa: F401

# ============================================================
# 1. IMPORT YOUR MODULES
# ============================================================
from environment_set_up.N_env_setup import make_sb_env, make_eval_sb_env
from environment_set_up.N_env_builders import get_cj_env, set_env_globals

# ============================================================
# 2. DEFINE TRIALS (rho=0, varying rho_c)
# ============================================================
trial_configs = [
    {"trial": "PPO_2Assets_Trial13", "rho_c": 1.0},
    {"trial": "PPO_2Assets_Trial14", "rho_c": 2.0},
    {"trial": "PPO_2Assets_Trial15", "rho_c": 3.0},
    {"trial": "PPO_2Assets_Trial16", "rho_c": 4.0},
    {"trial": "PPO_2Assets_Trial17", "rho_c": 5.0},
]

checkpoint_root = os.path.join(PROJECT_ROOT, "N_SB_models/PPO_Checkpoints_2Assets")

# ============================================================
# 3. GLOBAL SETTINGS
# ============================================================
terminal_time = 1.0
n_steps = 100
num_assets = 2
num_trajectories = 1000

train_seed = 123
network_seed = 123
eval_seed = 456

eval_phi = 0.0
eval_alpha = 0.2

n_eval_episodes = 100000
deterministic = True

# ============================================================
# 4. HELPERS
# ============================================================
def load_model_robust(path, env):
    try:
        model = PPO.load(path, env=env, device="cpu")
        print("Model loaded successfully.")
        return model
    except Exception as e:
        print("Standard load failed. Retrying with custom_objects...")
        print(e)
        return PPO.load(
            path,
            env=env,
            device="cpu",
            custom_objects={
                "learning_rate": 0.0,
                "lr_schedule": lambda _: 0.0,
                "clip_range": lambda _: 0.2,
            },
        )


def collect_episode_rewards(model, vec_env, n_episodes):
    rewards = []
    obs = vec_env.reset()

    while len(rewards) < n_episodes:
        actions, _ = model.predict(obs, deterministic=True)
        obs, _, dones, infos = vec_env.step(actions)

        for done, info in zip(dones, infos):
            if done:
                rewards.append(float(info["episode"]["r"]))

    return np.array(rewards[:n_episodes])


def summarize(x):
    return {
        "n_episodes": int(len(x)),
        "mean": float(np.mean(x)),
        "variance_sample": float(np.var(x, ddof=1)),
        "std_sample": float(np.std(x, ddof=1)),
        "min": float(np.min(x)),
        "max": float(np.max(x)),
        "q01": float(np.quantile(x, 0.01)),
        "q05": float(np.quantile(x, 0.05)),
        "q25": float(np.quantile(x, 0.25)),
        "q50": float(np.quantile(x, 0.50)),
        "q75": float(np.quantile(x, 0.75)),
        "q95": float(np.quantile(x, 0.95)),
        "q99": float(np.quantile(x, 0.99)),
    }


# ============================================================
# 5. LOOP OVER TRIALS
# ============================================================
for cfg in trial_configs:

    trial_folder = cfg["trial"]
    rho_c = cfg["rho_c"]

    print(f"\n==============================")
    print(f" Running {trial_folder} | rho_c={rho_c}")
    print(f"==============================")

    # ========================================================
    # MODEL PATH
    # ========================================================
    model_path = os.path.join(
        checkpoint_root,
        trial_folder,
        f"{trial_folder}_100M.zip"
    )

    run_tag = f"{trial_folder}_100M"

    # ========================================================
    # SAVE PATHS (UPDATED HERE)
    # ========================================================
    fig_root = os.path.join(PROJECT_ROOT, "N_figures/2Assets/PnL_LOB_Graph")
    data_root = os.path.join(PROJECT_ROOT, "N_figures/2Assets/PnL_LOB_Data")

    fig_dir = os.path.join(fig_root, trial_folder)
    data_dir = os.path.join(data_root, trial_folder)

    os.makedirs(fig_dir, exist_ok=True)
    os.makedirs(data_dir, exist_ok=True)

    print("\nSaving under:")
    print("  fig_dir :", fig_dir)
    print("  data_dir:", data_dir)

    # ========================================================
    # ENV SETTINGS
    # ========================================================
    set_env_globals(
        terminal_time=terminal_time,
        n_steps=n_steps,
        phi=eval_phi,
        alpha=eval_alpha,
        train_seed=train_seed,
    )

    cross_asset_influence = [[0.0, 0.0],
                             [0.0, 0.0]]

    c_cross_asset_influence = [[0.0, rho_c],
                               [rho_c, 0.0]]

    env_kwargs = dict(
        cross_asset_influence=cross_asset_influence,
        c_cross_asset_influence=c_cross_asset_influence,
    )

    # ========================================================
    # BUILD ENV
    # ========================================================
    train_bundle = make_sb_env(
        get_env_fn=get_cj_env,
        num_trajectories=num_trajectories,
        num_assets=num_assets,
        env_seed=train_seed,
        **env_kwargs,
    )

    eval_bundle = make_eval_sb_env(
        get_env_fn=get_cj_env,
        train_bundle=train_bundle,
        eval_seed=eval_seed,
        **env_kwargs,
    )

    sb_eval_env = VecMonitor(eval_bundle.sb_eval_env)

    # ========================================================
    # LOAD MODEL
    # ========================================================
    model = load_model_robust(model_path, sb_eval_env)

    # ========================================================
    # RUN EVAL
    # ========================================================
    rewards = collect_episode_rewards(
        model, sb_eval_env, n_eval_episodes
    )

    summary = summarize(rewards)

    # ========================================================
    # PRINT SUMMARY
    # ========================================================
    print("\n===== TERMINAL CUMULATIVE REWARD SUMMARY =====")
    for k, v in summary.items():
        if isinstance(v, float):
            print(f"{k:>18}: {v:.8f}")
        else:
            print(f"{k:>18}: {v}")

    # ========================================================
    # SAVE DATA
    # ========================================================
    base_name = f"{run_tag}_PnL"

    rewards_txt_path = os.path.join(data_dir, f"{base_name}.txt")
    summary_json_path = os.path.join(data_dir, f"{base_name}_summary.json")
    hist_png_path = os.path.join(fig_dir, f"{base_name}.png")

    np.savetxt(rewards_txt_path, rewards)

    with open(summary_json_path, "w") as f:
        json.dump(summary, f, indent=2)

    # ========================================================
    # PLOT
    # ========================================================
    plt.figure(figsize=(8, 5))
    plt.hist(rewards, bins=60)
    plt.title(f"{trial_folder} (rho_c={rho_c})")
    plt.xlabel("Terminal Reward")
    plt.ylabel("Frequency")
    plt.grid(alpha=0.3)

    plt.savefig(hist_png_path, dpi=600)
    plt.close()

    # ========================================================
    # FINAL PRINTS
    # ========================================================
    print("\nSaved:")
    print("  Histogram PNG :", hist_png_path)
    print("  Rewards TXT   :", rewards_txt_path)
    print("  Summary JSON  :", summary_json_path)

In [ ]:
import os
PROJECT_ROOT = os.environ.get("MBT_PROJECT_ROOT")
if not PROJECT_ROOT:
    PROJECT_ROOT = os.getcwd()
    while not os.path.isdir(os.path.join(PROJECT_ROOT, "mbt_gym")) and os.path.dirname(PROJECT_ROOT) != PROJECT_ROOT:
        PROJECT_ROOT = os.path.dirname(PROJECT_ROOT)

# LOB
# Multiple MODELs → MULTIPLE ENVIRONMENTS
# Models are trained (previously) with different values of rho_c={1,2,3,4,5} while rho=0
# And evaluated in their respective environments in the previous code (basically simulations) 
# Here we are just loading pnl data (from previous simulations) and plotting pnl metrics

# Combined Results: 
# Terminal PnL Distributions Plotted Together
# Mean, SD and Sharp Ratio of PnL is plotted over rho_c
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# ============================================================
# GLOBAL STYLE
# ============================================================
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman", "CMU Serif", "DejaVu Serif"],
    "mathtext.fontset": "cm",
    "axes.labelsize": 26,
    "xtick.labelsize": 20,
    "ytick.labelsize": 20,
    "axes.linewidth": 1.2,
    "lines.linewidth": 6,
})

# ============================================================
# 1. SETTINGS
# ============================================================
assets_tag = "2Assets"

base_data_root = os.path.join(PROJECT_ROOT, f"N_figures/{assets_tag}/PnL_LOB_Data")
base_fig_root  = os.path.join(PROJECT_ROOT, f"N_figures/{assets_tag}/PnL_LOB_Graph/Combined_Trials")

tests_root = os.path.join(base_fig_root, "tests")

os.makedirs(base_fig_root, exist_ok=True)
os.makedirs(tests_root, exist_ok=True)

# ============================================================
# 2. MAP rho_c → FILE
# ============================================================
rho_map = {
    1.0: ("PPO_2Assets_Trial13", "PPO_2Assets_Trial13_100M"),
    2.0: ("PPO_2Assets_Trial14", "PPO_2Assets_Trial14_100M"),
    3.0: ("PPO_2Assets_Trial15", "PPO_2Assets_Trial15_100M"),
    4.0: ("PPO_2Assets_Trial16", "PPO_2Assets_Trial16_100M"),
    5.0: ("PPO_2Assets_Trial17", "PPO_2Assets_Trial17_100M"),
}

rho_values = sorted(rho_map.keys())

file_paths = []
labels = []

for rho in rho_values:
    trial_folder, run_tag = rho_map[rho]

    file_path = os.path.join(
        base_data_root,
        trial_folder,
        f"{run_tag}_PnL.txt"
    )

    file_paths.append(file_path)
    labels.append(rho)

print("Files to load:")
for p in file_paths:
    print(" ", p)

# ============================================================
# 3. LOAD DATA + COMPUTE STATS
# ============================================================
all_rewards = []
summary_rows = []

print("\n===== SUMMARY BY rho_c =====\n")

for file_path, rho in zip(file_paths, labels):

    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Missing file: {file_path}")

    rewards = np.loadtxt(file_path, dtype=float)

    if len(rewards) < 2:
        raise ValueError(f"Not enough observations in file: {file_path}")

    all_rewards.append(rewards)

    mean_val = np.mean(rewards)
    var_val  = np.var(rewards, ddof=1)
    std_val  = np.sqrt(var_val)

    sharpe_val = mean_val / std_val if std_val > 0 else np.nan

    summary_rows.append({
        "rho_c": float(rho),
        "mean": float(mean_val),
        "variance": float(var_val),
        "std": float(std_val),
        "sharpe": float(sharpe_val) if not np.isnan(sharpe_val) else None,
        "n": int(len(rewards)),
    })

    print(
        f"rho_c = {rho:.2f}   "
        f"n = {len(rewards):d}   "
        f"mean = {mean_val:.8f}   "
        f"variance = {var_val:.8f}   "
        f"std = {std_val:.8f}   "
        f"Sharpe = {sharpe_val:.8f}"
    )

summary_df = pd.DataFrame(summary_rows)

summary_csv = os.path.join(base_fig_root, "PnL_LOB_summary_by_rho.csv")
summary_json = os.path.join(base_fig_root, "PnL_LOB_summary_by_rho.json")

summary_df.to_csv(summary_csv, index=False)
summary_df.to_json(summary_json, orient="records", indent=4)

print("\nSaved summary table:")
print(" ", summary_csv)
print(" ", summary_json)

# ============================================================
# 4. COMMON BINS
# ============================================================
global_min = min(np.min(x) for x in all_rewards)
global_max = max(np.max(x) for x in all_rewards)

n_bins = 80
bins = np.linspace(global_min, global_max, n_bins + 1)

# slight tightening only
#margin = 0.02 * (global_max - global_min)
margin = 0.00 * (global_max - global_min)

# ============================================================
# 5. HISTOGRAM OVERLAY
# ============================================================
plt.figure(figsize=(10, 6))

for rewards, rho in zip(all_rewards, labels):
    plt.hist(rewards, bins=bins, alpha=0.35, density=True,
             label=rf"$\rho_c = {int(rho)}$") # label=rf"$\rho_c = {rho:.2f}$"

plt.xlabel("Terminal PnL")
plt.ylabel("Density")
plt.legend(fontsize=20)

plt.grid(True, linestyle="--", linewidth=0.8, alpha=0.7)
plt.tick_params(width=1.5, length=6)

plt.xlim(global_min + margin, global_max - margin)

plt.tight_layout()

overlay_pdf = os.path.join(base_fig_root, "PnL_LOB_hist_overlay.pdf")
overlay_png = os.path.join(base_fig_root, "PnL_LOB_hist_overlay.png")

plt.savefig(overlay_pdf, bbox_inches="tight")
plt.savefig(overlay_png, dpi=600, bbox_inches="tight")

print("\nSaved histogram overlay:")
print(" ", overlay_pdf)
print(" ", overlay_png)

plt.show()

# ============================================================
# 6. LINE HISTOGRAM
# ============================================================
plt.figure(figsize=(10, 6))

for rewards, rho in zip(all_rewards, labels):
    counts, edges = np.histogram(rewards, bins=bins, density=True)
    centers = 0.5 * (edges[:-1] + edges[1:])
    plt.plot(centers, counts, label=rf"$\rho_c = {rho:.2f}$")

plt.xlabel("Terminal PnL")
plt.ylabel("Relative Frequency")
plt.legend(fontsize=12)

plt.grid(True, linestyle="--", linewidth=0.8, alpha=0.7)
plt.tick_params(width=1.5, length=6)

plt.xlim(global_min + margin, global_max - margin)

plt.tight_layout()

line_pdf = os.path.join(base_fig_root, "PnL_LOB_hist_lines.pdf")
line_png = os.path.join(base_fig_root, "PnL_LOB_hist_lines.png")

plt.savefig(line_pdf, bbox_inches="tight")
plt.savefig(line_png, dpi=600, bbox_inches="tight")

print("\nSaved line histogram:")
print(" ", line_pdf)
print(" ", line_png)

plt.show()

# ============================================================
# 7. MEAN PLOT
# ============================================================
params = summary_df["rho_c"].to_numpy()
means = summary_df["mean"].to_numpy()

plt.figure(figsize=(8.5, 5.5))
plt.plot(params, means, "o-")

plt.xticks([1, 2, 3, 4, 5])

plt.xlabel(r"Cross-asset influence parameter ($\rho_c$)")
plt.ylabel("Mean Terminal PnL")

plt.grid(True, linestyle="--", linewidth=0.8, alpha=0.7)
plt.tick_params(width=1.5, length=6)

plt.tight_layout()

path_pdf = os.path.join(base_fig_root, "PnL_LOB_mean.pdf")
path_png = os.path.join(base_fig_root, "PnL_LOB_mean.png")

plt.savefig(path_pdf, bbox_inches="tight")
plt.savefig(path_png, dpi=600, bbox_inches="tight")

print("\nSaved mean plot:")
print(" ", path_pdf)
print(" ", path_png)

plt.show()

# ============================================================
# 8. STD PLOT
# ============================================================
stds = summary_df["std"].to_numpy()

plt.figure(figsize=(8.5, 5.5))
plt.plot(params, stds, "o-")

plt.xticks([1, 2, 3, 4, 5])

plt.xlabel(r"Cross-asset influence parameter ($\rho_c$)")
plt.ylabel("Standard deviation of Terminal PnL")

plt.grid(True, linestyle="--", linewidth=0.8, alpha=0.7)
plt.tick_params(width=1.5, length=6)

plt.tight_layout()

path_pdf = os.path.join(base_fig_root, "PnL_LOB_sd.pdf")
path_png = os.path.join(base_fig_root, "PnL_LOB_sd.png")

plt.savefig(path_pdf, bbox_inches="tight")
plt.savefig(path_png, dpi=600, bbox_inches="tight")

print("\nSaved std plot:")
print(" ", path_pdf)
print(" ", path_png)

plt.show()

# ============================================================
# 8.5 COMBINED MEAN + STD
# ============================================================
fig, ax1 = plt.subplots(figsize=(9, 5.5))

ax1.plot(params, means, "o-", color="blue", label="Mean", ms=15, linewidth=6)
ax1.set_xlabel(r"Cross-asset influence ($\rho_c$)")
ax1.set_ylabel("Mean")

ax1.set_xticks([1, 2, 3, 4, 5])
ax1.tick_params(width=1.5, length=6)

ax1.grid(True, linestyle="--", linewidth=0.8, alpha=0.7)

ax2 = ax1.twinx()
ax2.plot(params, stds, "s--", color="red", label="SD", ms=15, linewidth=6)
ax2.set_ylabel("Standard Deviation")
ax2.tick_params(width=1.5, length=6)

lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
ax1.legend(lines_1 + lines_2, labels_1 + labels_2, fontsize=16)

plt.tight_layout()

combined_pdf = os.path.join(base_fig_root, "PnL_LOB_mean_sd_combined.pdf")
combined_png = os.path.join(base_fig_root, "PnL_LOB_mean_sd_combined.png")

plt.savefig(combined_pdf, bbox_inches="tight")
plt.savefig(combined_png, dpi=600, bbox_inches="tight")

print("\nSaved combined mean + std plot:")
print(" ", combined_pdf)
print(" ", combined_png)

plt.show()

# ============================================================
# 9. SHARPE PLOT
# ============================================================
sharpes = summary_df["sharpe"].to_numpy()

plt.figure(figsize=(9, 5.5))
plt.plot(params, sharpes, "o-", color="green", ms=15, linewidth=6)

plt.xticks([1, 2, 3, 4, 5])

plt.xlabel(r"Cross-asset influence ($\rho_c$)")
plt.ylabel("Sharpe Ratio")

plt.grid(True, linestyle="--", linewidth=0.8, alpha=0.7)
plt.tick_params(width=1.5, length=6)

plt.tight_layout()

path_pdf = os.path.join(base_fig_root, "PnL_LOB_sharpe.pdf")
path_png = os.path.join(base_fig_root, "PnL_LOB_sharpe.png")

plt.savefig(path_pdf, bbox_inches="tight")
plt.savefig(path_png, dpi=600, bbox_inches="tight")

print("\nSaved sharpe plot:")
print(" ", path_pdf)
print(" ", path_png)

plt.show()
# Here here here

In [ ]:
# ============================================================
# NOTE: Being evauluated here. Do not Rerun.

# LOB : COST of IGNORANCE : what happens when rho_c exists but MM ignores them 
# FIXED MODEL → MULTIPLE ENVIRONMENTS (LOB COUPLING)
# Model trained once (rho=0, rho_c=0)
# Evaluate under different rho_c values (in this code, evaluations being run)
# rho_c = {1,2,3,4,5}, rho = 0
# ============================================================

import os
import sys
import json
import numpy as np
import matplotlib.pyplot as plt

from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import VecMonitor

# ============================================================
# 0. PROJECT PATH
# ============================================================
import os
PROJECT_ROOT = os.environ.get("MBT_PROJECT_ROOT")
if not PROJECT_ROOT:
    PROJECT_ROOT = os.getcwd()
    while not os.path.isdir(os.path.join(PROJECT_ROOT, "mbt_gym")) and os.path.dirname(PROJECT_ROOT) != PROJECT_ROOT:
        PROJECT_ROOT = os.path.dirname(PROJECT_ROOT)

if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

import mbt_gym  # noqa

# ============================================================
# 1. IMPORT YOUR MODULES
# ============================================================
from environment_set_up.N_env_setup import make_sb_env, make_eval_sb_env
from environment_set_up.N_env_builders import get_cj_env, set_env_globals

# ============================================================
# 2. FIXED MODEL (TRAINED ONCE)
# ============================================================
trial_folder = "PPO_2Assets_Trial0"
run_tag = "PPO_2Assets_Trial0_100M"

checkpoint_root = os.path.join(PROJECT_ROOT, "N_SB_models/PPO_Checkpoints_2Assets")

def build_checkpoint_path(root, trial_folder, run_tag):
    return os.path.join(root, trial_folder, f"{run_tag}.zip")

model_path = build_checkpoint_path(checkpoint_root, trial_folder, run_tag)

if not os.path.exists(model_path):
    raise FileNotFoundError(f"Model not found: {model_path}")

print(f"Using model: {model_path}")

# ============================================================
# 3. EVALUATION ENVIRONMENTS (rho_c VALUES)
# ============================================================
rho_c_eval_list = [1.0, 2.0, 3.0, 4.0, 5.0]

# ============================================================
# 4. SETTINGS
# ============================================================
terminal_time = 1.0
n_steps = 100
num_trajectories = 1000

train_seed = 123
eval_seed = 456

eval_phi = 0.0
eval_alpha = 0.2

n_eval_episodes = 100000
deterministic = True

# ============================================================
# 5. OTHER PARAMS (UNCHANGED)
# ============================================================
baseline_arrival_rate = None
mean_reversion_speed = None
self_jump_size = None
mutual_jump_size = None
synchrony_factor = None

c_baseline_depth = None
c_mean_reversion_speed = None
c_self_jump_size = None
c_mutual_jump_size = None
c_synchrony_factor = None

sigma = None
initial_price = None
initial_inventory = None

# ============================================================
# 6. SAVE DIR (NEW FOLDERS)
# ============================================================
assets_tag = "2Assets"

def build_save_dirs(rho_c):
    fig_root = os.path.join(PROJECT_ROOT, f"N_figures/{assets_tag}/PnL_Graph_LOB_FixedModelEval")
    data_root = os.path.join(PROJECT_ROOT, f"N_figures/{assets_tag}/PnL_Data_LOB_FixedModelEval")

    fig_dir = os.path.join(fig_root, f"rho_c_{rho_c}")
    data_dir = os.path.join(data_root, f"rho_c_{rho_c}")

    os.makedirs(fig_dir, exist_ok=True)
    os.makedirs(data_dir, exist_ok=True)

    return fig_dir, data_dir

# ============================================================
# 7. LOAD MODEL
# ============================================================
def load_model_robust(path, env):
    try:
        return PPO.load(path, env=env, device="cpu")
    except Exception:
        return PPO.load(
            path,
            env=env,
            device="cpu",
            custom_objects={
                "learning_rate": 0.0,
                "lr_schedule": lambda _: 0.0,
                "clip_range": lambda _: 0.2,
            },
        )

# ============================================================
# 8. EVAL FUNCTION
# ============================================================
def collect_rewards(model, vec_env, n_episodes):
    rewards = []
    obs = vec_env.reset()

    while len(rewards) < n_episodes:
        actions, _ = model.predict(obs, deterministic=True)
        obs, _, dones, infos = vec_env.step(actions)

        for done, info in zip(dones, infos):
            if done:
                rewards.append(float(info["episode"]["r"]))

    return np.array(rewards[:n_episodes])

def summarize(x):
    return {
        "mean": float(np.mean(x)),
        "std": float(np.std(x, ddof=1)),
        "q05": float(np.quantile(x, 0.05)),
        "q50": float(np.quantile(x, 0.50)),
        "q95": float(np.quantile(x, 0.95)),
    }

# ============================================================
# 9. MAIN LOOP (OVER rho_c)
# ============================================================
all_results = {}

for rho_c in rho_c_eval_list:

    print("\n" + "="*60)
    print(f"Evaluating environment with LOB cross-asset influence rho_c = {rho_c}")
    print("="*60)

    # --------------------------------------------------------
    # FIX rho = 0
    # --------------------------------------------------------
    eval_cross = np.zeros((2, 2))  # rho = 0

    # --------------------------------------------------------
    # VARY rho_c
    # --------------------------------------------------------
    eval_c_cross = np.array([[0.0, rho_c], [rho_c, 0.0]])

    fig_dir, data_dir = build_save_dirs(rho_c)

    # --------------------------------------------------------
    # ENV KWARGS
    # --------------------------------------------------------
    env_kwargs = dict(
        cross_asset_influence=eval_cross,
        c_cross_asset_influence=eval_c_cross,
    )

    # --------------------------------------------------------
    # BUILD ENV
    # --------------------------------------------------------
    set_env_globals(
        terminal_time=terminal_time,
        n_steps=n_steps,
        phi=eval_phi,
        alpha=eval_alpha,
        train_seed=train_seed,
    )

    train_bundle = make_sb_env(
        get_env_fn=get_cj_env,
        num_trajectories=num_trajectories,
        num_assets=2,
        initial_inventory=initial_inventory,
        do_reward_scaling=False,
        env_seed=train_seed,
        **env_kwargs,
    )

    eval_bundle = make_eval_sb_env(
        get_env_fn=get_cj_env,
        train_bundle=train_bundle,
        eval_seed=eval_seed,
        **env_kwargs,
    )

    sb_env = VecMonitor(eval_bundle.sb_eval_env)

    # --------------------------------------------------------
    # LOAD MODEL
    # --------------------------------------------------------
    model = load_model_robust(model_path, sb_env)

    # --------------------------------------------------------
    # RUN EVAL
    # --------------------------------------------------------
    rewards = collect_rewards(model, sb_env, n_eval_episodes)
    summary = summarize(rewards)

    print("Mean:", summary["mean"], "Std:", summary["std"])

    # --------------------------------------------------------
    # SAVE DATA
    # --------------------------------------------------------
    base_name = f"{run_tag}_rho_c{rho_c}"

    np.savetxt(os.path.join(data_dir, f"{base_name}.csv"), rewards)

    with open(os.path.join(data_dir, f"{base_name}_summary.json"), "w") as f:
        json.dump(summary, f, indent=2)

    # --------------------------------------------------------
    # HISTOGRAM
    # --------------------------------------------------------
    plt.figure(figsize=(8,5))
    plt.hist(rewards, bins=60)
    plt.title(f"PnL — Fixed Model | rho_c={rho_c}")
    plt.grid(True)
    plt.tight_layout()

    plt.savefig(os.path.join(fig_dir, f"{base_name}.pdf"))
    plt.close()

    all_results[rho_c] = summary

# ============================================================
# 10. COMBINED SUMMARY
# ============================================================
combined_path = os.path.join(PROJECT_ROOT, f"N_figures/{assets_tag}/PnL_Data_LOB_FixedModelEval/combined.json")

with open(combined_path, "w") as f:
    json.dump(all_results, f, indent=2)

print("\nDONE. Combined summary saved.")

In [ ]:
import os
PROJECT_ROOT = os.environ.get("MBT_PROJECT_ROOT")
if not PROJECT_ROOT:
    PROJECT_ROOT = os.getcwd()
    while not os.path.isdir(os.path.join(PROJECT_ROOT, "mbt_gym")) and os.path.dirname(PROJECT_ROOT) != PROJECT_ROOT:
        PROJECT_ROOT = os.path.dirname(PROJECT_ROOT)

# ============================================================
# LOB : COST of IGNORANCE : what happens when rho_c exists but MM ignores them 
# FIXED MODEL → MULTIPLE ENVIRONMENTS (LOB COUPLING)
# Model trained once (rho=0, rho_c=0)
# Evaluate under different rho_c values (in this code, evaluations being run)
# rho_c = {1,2,3,4,5}, rho = 0

# Here only COMBINED DISTRIBUTION being plotted using already saved data
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# GLOBAL STYLE (CONSISTENT)
# ============================================================
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman", "CMU Serif", "DejaVu Serif"],
    "mathtext.fontset": "cm",
    "axes.labelsize": 18,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "axes.linewidth": 1.2,
    "lines.linewidth": 3.5,
})

# ============================================================
# PATHS
# ============================================================
assets_tag = "2Assets"

data_root = os.path.join(PROJECT_ROOT, f"N_figures/{assets_tag}/PnL_Data_LOB_FixedModelEval")

save_root = os.path.join(PROJECT_ROOT, f"N_figures/{assets_tag}/PnL_Data_FixedModelEval/combined_trials")

os.makedirs(save_root, exist_ok=True)

# ============================================================
# SETTINGS
# ============================================================
rho_c_list = [1.0, 2.0, 3.0, 4.0, 5.0]
run_tag = "PPO_2Assets_Trial0_100M"

# ============================================================
# STORAGE
# ============================================================
all_rewards = []
summary_rows = []

# ============================================================
# LOAD DATA
# ============================================================
for rho_c in rho_c_list:

    file_path = os.path.join(
        data_root,
        f"rho_c_{rho_c}",
        f"{run_tag}_rho_c{rho_c}.csv"
    )

    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Missing file: {file_path}")

    rewards = np.loadtxt(file_path)

    n = len(rewards)
    mean = np.mean(rewards)
    var = np.var(rewards, ddof=1)
    std = np.sqrt(var)
    sharpe = mean / std if std > 0 else np.nan

    all_rewards.append(rewards)

    summary_rows.append({
        "rho_c": rho_c,
        "mean": mean,
        "variance": var,
        "std": std,
        "sharpe": sharpe,
        "n": n
    })

    print(f"Loaded rho_c={rho_c:.1f} | n={n}")

# ============================================================
# SAVE SUMMARY
# ============================================================
summary_df = pd.DataFrame(summary_rows)

csv_path = os.path.join(save_root, "PnL_summary_by_rho_c.csv")
json_path = os.path.join(save_root, "PnL_summary_by_rho_c.json")

summary_df.to_csv(csv_path, index=False)
summary_df.to_json(json_path, orient="records", indent=4)

print("\nSaved summary:")
print(csv_path)
print(json_path)

# ============================================================
# GLOBAL RANGE (NO TRIMMING)
# ============================================================
global_min = min(np.min(x) for x in all_rewards)
global_max = max(np.max(x) for x in all_rewards)

bins = np.linspace(global_min, global_max, 81)

# ============================================================
# HISTOGRAM OVERLAY
# ============================================================
plt.figure(figsize=(10,6))

for rewards, rho_c in zip(all_rewards, rho_c_list):
    plt.hist(
        rewards,
        bins=bins,
        alpha=0.35,
        density=True,
        label=f"$\\rho_c$={rho_c:.1f}"
    )

plt.xlabel("Terminal PnL")
plt.ylabel("Relative Frequency")
plt.legend(fontsize=12)
plt.grid(True, linestyle="--", linewidth=0.8, alpha=0.7)
plt.tick_params(width=1.5, length=6)

plt.tight_layout()

overlay_pdf = os.path.join(save_root, "PnL_histogram_by_rho_c_overlay.pdf")
overlay_png = os.path.join(save_root, "PnL_histogram_by_rho_c_overlay.png")

plt.savefig(overlay_pdf, bbox_inches="tight")
plt.savefig(overlay_png, dpi=600, bbox_inches="tight")

print("\nSaved overlay histogram:")
print(overlay_pdf)
print(overlay_png)

plt.show()

# ============================================================
# LINE HISTOGRAM
# ============================================================
plt.figure(figsize=(10,6))

for rewards, rho_c in zip(all_rewards, rho_c_list):
    counts, edges = np.histogram(rewards, bins=bins, density=True)
    centers = 0.5 * (edges[:-1] + edges[1:])
    plt.plot(centers, counts, label=f"$\\rho_c$={rho_c:.1f}")

plt.xlabel("Terminal PnL")
plt.ylabel("Relative Frequency")
plt.legend(fontsize=12)
plt.grid(True, linestyle="--", linewidth=0.8, alpha=0.7)
plt.tick_params(width=1.5, length=6)

plt.tight_layout()

lines_pdf = os.path.join(save_root, "PnL_histogram_by_rho_c_lines.pdf")
lines_png = os.path.join(save_root, "PnL_histogram_by_rho_c_lines.png")

plt.savefig(lines_pdf, bbox_inches="tight")
plt.savefig(lines_png, dpi=600, bbox_inches="tight")

print("\nSaved line histogram:")
print(lines_pdf)
print(lines_png)

plt.show()

In [ ]:
import os
PROJECT_ROOT = os.environ.get("MBT_PROJECT_ROOT")
if not PROJECT_ROOT:
    PROJECT_ROOT = os.getcwd()
    while not os.path.isdir(os.path.join(PROJECT_ROOT, "mbt_gym")) and os.path.dirname(PROJECT_ROOT) != PROJECT_ROOT:
        PROJECT_ROOT = os.path.dirname(PROJECT_ROOT)

# ============================================================
# LOB COMPARISON: A vs B 
# (A) Adaptive Policy (trained per rho_c) and evaluated in an environments with different rho_c 
# (B) Fixed Policy (trained at rho_c=0) and evaluated in an environments with different rho_c   
# Just Trying to Show Cost of Ignorance 
# Meand, SD, Sharp Ratio over rho_c 
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# GLOBAL STYLE (CONSISTENT WITH OTHER FILES)
# ============================================================
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman", "CMU Serif", "DejaVu Serif"],
    "mathtext.fontset": "cm",
    "axes.labelsize": 26, #18
    "xtick.labelsize": 20,
    "ytick.labelsize": 20,
    "axes.linewidth": 1.2,
    "lines.linewidth": 6,
})

# ============================================================
# 1. SETTINGS
# ============================================================
assets_tag = "2Assets"

# -------- SAVE ROOT (NEW) --------
fig_root = os.path.join(PROJECT_ROOT, f"N_figures/{assets_tag}/PnL_Graph_LOB_Fixed_Adaptive_Comparison")
os.makedirs(fig_root, exist_ok=True)

# -------- (A) MULTI-MODEL DATA --------
base_data_root_A = os.path.join(PROJECT_ROOT, f"N_figures/{assets_tag}/PnL_LOB_Data")

rho_map = {
    1.0: ("PPO_2Assets_Trial13", "PPO_2Assets_Trial13_100M"),
    2.0: ("PPO_2Assets_Trial14", "PPO_2Assets_Trial14_100M"),
    3.0: ("PPO_2Assets_Trial15", "PPO_2Assets_Trial15_100M"),
    4.0: ("PPO_2Assets_Trial16", "PPO_2Assets_Trial16_100M"),
    5.0: ("PPO_2Assets_Trial17", "PPO_2Assets_Trial17_100M"),
}

# -------- (B) FIXED MODEL DATA --------
base_data_root_B = os.path.join(PROJECT_ROOT, f"N_figures/{assets_tag}/PnL_Data_LOB_FixedModelEval")

rho_values = sorted(rho_map.keys())

# ============================================================
# 2. LOAD DATASET A (ADAPTIVE POLICY)
# ============================================================
means_A, stds_A, sharpes_A = [], [], []
summary_A = []

for rho in rho_values:

    trial_folder, run_tag = rho_map[rho]

    file_path = os.path.join(
        base_data_root_A,
        trial_folder,
        f"{run_tag}_PnL.txt"
    )

    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Missing file A: {file_path}")

    rewards = np.loadtxt(file_path)

    n = len(rewards)
    mean = np.mean(rewards)
    std  = np.std(rewards, ddof=1)
    var  = std**2
    sharpe = mean / std if std > 0 else np.nan

    means_A.append(mean)
    stds_A.append(std)
    sharpes_A.append(sharpe)

    summary_A.append({
        "rho_c": rho,
        "policy": "Adaptive",
        "n": n,
        "mean": mean,
        "std": std,
        "variance": var,
        "sharpe": sharpe if not np.isnan(sharpe) else None
    })

# ============================================================
# 3. LOAD DATASET B (FIXED POLICY)
# ============================================================
means_B, stds_B, sharpes_B = [], [], []
summary_B = []

for rho_c in rho_values:

    data_dir = os.path.join(base_data_root_B, f"rho_c_{rho_c}")

    files = [f for f in os.listdir(data_dir) if f.endswith(".csv")]

    if len(files) == 0:
        raise FileNotFoundError(f"No CSV found in {data_dir}")

    file_path = os.path.join(data_dir, files[0])

    rewards = np.loadtxt(file_path)

    n = len(rewards)
    mean = np.mean(rewards)
    std  = np.std(rewards, ddof=1)
    var  = std**2
    sharpe = mean / std if std > 0 else np.nan

    means_B.append(mean)
    stds_B.append(std)
    sharpes_B.append(sharpe)

    summary_B.append({
        "rho_c": rho_c,
        "policy": "Fixed",
        "n": n,
        "mean": mean,
        "std": std,
        "variance": var,
        "sharpe": sharpe if not np.isnan(sharpe) else None
    })

# ============================================================
# 4. COMBINE + SAVE SUMMARY
# ============================================================
summary_df = pd.DataFrame(summary_A + summary_B)

summary_csv = os.path.join(fig_root, "PnL_LOB_fixed_vs_adaptive_summary.csv")
summary_json = os.path.join(fig_root, "PnL_LOB_fixed_vs_adaptive_summary.json")

summary_df.to_csv(summary_csv, index=False)
summary_df.to_json(summary_json, orient="records", indent=4)
summary_excel = os.path.join(fig_root, "PnL_LOB_fixed_vs_adaptive_summary.xlsx")
summary_df.to_excel(summary_excel, index=False)

print(" ", summary_excel)

print("\nSaved summary:")
print(" ", summary_csv)
print(" ", summary_json)

# ============================================================
# 5. ARRAYS
# ============================================================
rho_vals = np.array(rho_values)

means_A = np.array(means_A)
stds_A = np.array(stds_A)
sharpes_A = np.array(sharpes_A)

means_B = np.array(means_B)
stds_B = np.array(stds_B)
sharpes_B = np.array(sharpes_B)

# ============================================================
# 6. MEAN PLOT
# ============================================================
plt.figure(figsize=(9, 5.5))

#plt.plot(rho_vals, means_A, "o-", label="Trained Policy")
#plt.plot(rho_vals, means_B, "s--", label="Fixed Policy")

plt.plot(rho_vals, means_A, "o-", markersize=15, label="Trained Policy")
plt.plot(rho_vals, means_B, "s--", markersize=15, label="Fixed Policy")

plt.xticks([1, 2, 3, 4, 5])

plt.xlabel(r"Cross-asset influence ($\rho_c$)")
plt.ylabel("Mean")

plt.grid(True, linestyle="--", linewidth=0.8, alpha=0.7)
plt.tick_params(width=1.5, length=6)

#plt.legend()
plt.legend(fontsize=16)
plt.tight_layout()

plt.savefig(os.path.join(fig_root, "mean_comparison.pdf"), bbox_inches="tight")
plt.savefig(os.path.join(fig_root, "mean_comparison.png"), dpi=600, bbox_inches="tight")

plt.show()

# ============================================================
# 7. STD PLOT
# ============================================================
plt.figure(figsize=(9, 5.5))

plt.plot(rho_vals, stds_A, "o-", label="Trained Policy")
plt.plot(rho_vals, stds_B, "s--", label="Fixed Policy")

plt.xticks([1, 2, 3, 4, 5])

plt.xlabel(r"Cross-asset influence parameter ($\rho_c$)")
plt.ylabel("Standard deviation of Terminal PnL")

plt.grid(True, linestyle="--", linewidth=0.8, alpha=0.7)
plt.tick_params(width=1.5, length=6)

plt.legend()
plt.tight_layout()

plt.savefig(os.path.join(fig_root, "std_comparison.pdf"), bbox_inches="tight")
plt.savefig(os.path.join(fig_root, "std_comparison.png"), dpi=600, bbox_inches="tight")

plt.show()

# ============================================================
# 8. SHARPE PLOT
# ============================================================
plt.figure(figsize=(9, 5.5))

plt.plot(rho_vals, sharpes_A, "o-", label="Trained Policy")
plt.plot(rho_vals, sharpes_B, "s--", label="Fixed Policy")

plt.xticks([1, 2, 3, 4, 5])

plt.xlabel(r"Cross-asset influence parameter ($\rho_c$)")
plt.ylabel("Sharpe ratio")

plt.grid(True, linestyle="--", linewidth=0.8, alpha=0.7)
plt.tick_params(width=1.5, length=6)

plt.legend()
plt.tight_layout()

plt.savefig(os.path.join(fig_root, "sharpe_comparison.pdf"), bbox_inches="tight")
plt.savefig(os.path.join(fig_root, "sharpe_comparison.png"), dpi=600, bbox_inches="tight")

plt.show()

In [ ]:
import os
PROJECT_ROOT = os.environ.get("MBT_PROJECT_ROOT")
if not PROJECT_ROOT:
    PROJECT_ROOT = os.getcwd()
    while not os.path.isdir(os.path.join(PROJECT_ROOT, "mbt_gym")) and os.path.dirname(PROJECT_ROOT) != PROJECT_ROOT:
        PROJECT_ROOT = os.path.dirname(PROJECT_ROOT)

# ============================================================
# MO COMPARISON: A vs B 
# (A) Adaptive Policy (trained per rho) and evaluated in an environments with different rho 
# (B) Fixed Policy (trained at rho=0) and evaluated in an environments with different rho   
# Just Trying to Show Cost of Ignorance 
# Meand, SD, Sharp Ratio over rho_c 
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# GLOBAL STYLE (IDENTICAL TO YOUR OTHER FILES)
# ============================================================
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman", "CMU Serif", "DejaVu Serif"],
    "mathtext.fontset": "cm",
    "axes.labelsize": 26, # 18
    "xtick.labelsize": 20, # 14
    "ytick.labelsize": 20, # 14
    "axes.linewidth": 1.2, # 1.2
    "lines.linewidth": 6, # 3.5
})

# ============================================================
# SETTINGS
# ============================================================
assets_tag = "2Assets"

#fig_root = os.path.join(PROJECT_ROOT, f"N_figures/{assets_tag}/PnL_Comparison_Final")
fig_root = os.path.join(PROJECT_ROOT, f"N_figures/{assets_tag}/PnL_Graph_MO_Fixed_Adaptive_Comparison")
os.makedirs(fig_root, exist_ok=True)

# ============================================================
# (A) TRAINED POLICY (SECOND CODE LOGIC)
# ============================================================
base_data_root_A = os.path.join(PROJECT_ROOT, f"N_figures/{assets_tag}/PnL_Data")

rho_map = {
    1.0: ("PPO_2Assets_Trial8",  "PPO_2Assets_Trial8_100M"),
    3.0: ("PPO_2Assets_Trial9",  "PPO_2Assets_Trial9_100M"),
    5.0: ("PPO_2Assets_Trial10", "PPO_2Assets_Trial10_100M"),
    7.0: ("PPO_2Assets_Trial11", "PPO_2Assets_Trial11_100M"),
    9.0: ("PPO_2Assets_Trial12", "PPO_2Assets_Trial12_100M"),
}

rho_vals = np.array(sorted(rho_map.keys()))

means_A, stds_A = [], []

for rho in rho_vals:

    trial_folder, run_tag = rho_map[rho]

    csv_path = os.path.join(
        base_data_root_A,
        trial_folder,
        f"{run_tag}_PnL_episode_rewards.csv"
    )

    if not os.path.exists(csv_path):
        raise FileNotFoundError(csv_path)

    df = pd.read_csv(csv_path)

    rewards = df["terminal_cumulative_reward"].dropna().to_numpy()

    means_A.append(np.mean(rewards))
    stds_A.append(np.std(rewards, ddof=1))

means_A = np.array(means_A)
stds_A  = np.array(stds_A)

# ============================================================
# (B) FIXED POLICY (FIRST CODE LOGIC)
# ============================================================
base_data_root_B = os.path.join(PROJECT_ROOT, f"N_figures/{assets_tag}/PnL_Data_FixedModelEval")

means_B, stds_B = [], []

for rho in rho_vals:

    data_dir = os.path.join(base_data_root_B, f"rho_{rho}")
    file_path = os.path.join(data_dir, f"PPO_2Assets_Trial0_100M_rho{rho}.csv")

    if not os.path.exists(file_path):
        raise FileNotFoundError(file_path)

    rewards = np.loadtxt(file_path)

    means_B.append(np.mean(rewards))
    stds_B.append(np.std(rewards, ddof=1))

means_B = np.array(means_B)
stds_B  = np.array(stds_B)

# ============================================================
# SAVE SUMMARY
# ============================================================
csv_summary_path = os.path.join(fig_root, "comparison_summary.csv")
json_summary_path = os.path.join(fig_root, "comparison_summary.json")

summary_df = pd.DataFrame({
    "rho": rho_vals,
    "mean_trained": means_A,
    "std_trained": stds_A,
    "mean_fixed": means_B,
    "std_fixed": stds_B
})

summary_df.to_csv(csv_summary_path, index=False)
summary_df.to_json(json_summary_path, orient="records", indent=4)
excel_summary_path = os.path.join(fig_root, "comparison_summary.xlsx")
summary_df.to_excel(excel_summary_path, index=False)
print(f"[SAVED] Excel summary → {excel_summary_path}")

print(f"[SAVED] CSV summary → {csv_summary_path}")
print(f"[SAVED] JSON summary → {json_summary_path}")

# ============================================================
# 1. MEAN COMPARISON
# ============================================================
plt.figure(figsize=(9,5.5))

plt.plot(rho_vals, means_A, "o-", markersize=15, label="Trained Policy")
plt.plot(rho_vals, means_B, "s--", markersize=15, label="Fixed Policy")

plt.xlabel(r"Cross-asset influence ($\rho$)")
plt.ylabel("Mean")

plt.xticks(rho_vals)

plt.grid(True, linestyle="--", linewidth=0.8, alpha=0.7)
plt.tick_params(width=1.5, length=6)

plt.legend(fontsize=16)

plt.tight_layout()

mean_pdf = os.path.join(fig_root, "mean_comparison.pdf")
mean_png = os.path.join(fig_root, "mean_comparison.png")

plt.savefig(mean_pdf, bbox_inches="tight")
plt.savefig(mean_png, dpi=600, bbox_inches="tight")

print(f"[SAVED] Mean plot (PDF) → {mean_pdf}")
print(f"[SAVED] Mean plot (PNG) → {mean_png}")

plt.show()

# ============================================================
# 2. STD COMPARISON
# ============================================================
plt.figure(figsize=(9,5.5))

plt.plot(rho_vals, stds_A, "o-", label="Trained Policy")
plt.plot(rho_vals, stds_B, "s--", label="Fixed Policy")

plt.xlabel(r"Cross-asset influence parameter ($\rho$)")
plt.ylabel("Standard Deviation")
#plt.ylabel("Standard deviation of Terminal PnL")

plt.xticks(rho_vals)

plt.grid(True, linestyle="--", linewidth=0.8, alpha=0.7)
plt.tick_params(width=1.5, length=6)

plt.legend(fontsize=12)

plt.tight_layout()

std_pdf = os.path.join(fig_root, "std_comparison.pdf")
std_png = os.path.join(fig_root, "std_comparison.png")

plt.savefig(std_pdf, bbox_inches="tight")
plt.savefig(std_png, dpi=600, bbox_inches="tight")

print(f"[SAVED] Std plot (PDF) → {std_pdf}")
print(f"[SAVED] Std plot (PNG) → {std_png}")

plt.show()

# ============================================================
# 3. COMBINED MEAN + STD (CLEAN VERSION)
# ============================================================
fig, ax1 = plt.subplots(figsize=(9,5.5))

# ---- MEAN ----
ax1.plot(rho_vals, means_A, "o-", color="blue", label="Mean (Trained)")
#ax1.plot(rho_vals, means_B, "o--", color="cyan", label="Mean (Fixed)")
ax1.plot(rho_vals, means_B,
         linestyle="--",
         marker="s",
         markersize=15,
         color="cyan",
         label="Mean (Fixed)")
ax1.set_xlabel(r"Cross-asset influence parameter ($\rho$)")
ax1.set_ylabel("Mean Terminal PnL")
ax1.tick_params(width=1.5, length=6)

ax1.grid(True, linestyle="--", linewidth=0.8, alpha=0.7)

# ---- STD ----
ax2 = ax1.twinx()
ax2.plot(rho_vals, stds_A, "s-", color="red", label="Std (Trained)")
ax2.plot(rho_vals, stds_B, "s--", color="orange", label="Std (Fixed)")
ax2.set_ylabel("Standard deviation of Terminal PnL")
ax2.tick_params(width=1.5, length=6)

# ---- LEGEND ----
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()

# Mean legend (tight spacing)
leg1 = ax1.legend(lines1, labels1,
                 fontsize=12,
                 loc="upper left",
                 bbox_to_anchor=(0,0.9),
                 handlelength=1.2,
                 handletextpad=0.4)

# Std legend (unchanged default look)
leg2 = ax1.legend(lines2, labels2,
                 fontsize=12,
                 loc="upper left",
                 bbox_to_anchor=(0,0.75))

ax1.add_artist(leg1)

plt.tight_layout()

combined_pdf = os.path.join(fig_root, "mean_std_combined.pdf")
combined_png = os.path.join(fig_root, "mean_std_combined.png")

plt.savefig(combined_pdf, bbox_inches="tight")
plt.savefig(combined_png, dpi=600, bbox_inches="tight")

print(f"[SAVED] Combined plot (PDF) → {combined_pdf}")
print(f"[SAVED] Combined plot (PNG) → {combined_png}")

plt.show()

In [ ]:
# ============================================================
# Both LOB and MO MODEL EVALUATION
# Models previously trained with:
#   rho = 5, rho_c = 3 : Being Evaluated (Here) in its Repective Environment
#  rho = 5, rho_c = 0 : Already Evaluated in its Repective Environmens (earlier and data was saved)
#  rho = 0, rho_c = 3 : Already Evaluated in its Repective Environmens (earlier and data was saved)
# ============================================================

import os
import sys
import json
import numpy as np
import matplotlib.pyplot as plt

from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import VecMonitor

# ============================================================
# 0. PROJECT PATH
# ============================================================
import os
PROJECT_ROOT = os.environ.get("MBT_PROJECT_ROOT")
if not PROJECT_ROOT:
    PROJECT_ROOT = os.getcwd()
    while not os.path.isdir(os.path.join(PROJECT_ROOT, "mbt_gym")) and os.path.dirname(PROJECT_ROOT) != PROJECT_ROOT:
        PROJECT_ROOT = os.path.dirname(PROJECT_ROOT)

if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

import mbt_gym  # noqa

# ============================================================
# 1. IMPORT MODULES
# ============================================================
from environment_set_up.N_env_setup import make_sb_env, make_eval_sb_env
from environment_set_up.N_env_builders import get_cj_env, set_env_globals

# ============================================================
# 2. MODEL PATH (Trial18)
# ============================================================
checkpoint_path = os.path.join(PROJECT_ROOT, "N_SB_models/PPO_Checkpoints_2Assets/PPO_2Assets_Trial18/PPO_2Assets_Trial18_100M.zip")

model_path = checkpoint_path
use_best_model = False

# ============================================================
# 3. INFER NAMES
# ============================================================
def infer_trial_folder(model_path):
    return os.path.basename(os.path.dirname(model_path))

def infer_assets_tag(trial_folder):
    return trial_folder.split("_")[1]

def infer_run_tag(model_path):
    return os.path.splitext(os.path.basename(model_path))[0]

trial_folder = infer_trial_folder(model_path)
assets_tag = infer_assets_tag(trial_folder)
run_tag = infer_run_tag(model_path)

print("trial_folder:", trial_folder)
print("assets_tag  :", assets_tag)
print("run_tag     :", run_tag)

# ============================================================
# 4. SAVE PATHS (NEW)
# ============================================================
fig_root = os.path.join(PROJECT_ROOT, f"N_figures/{assets_tag}/PnL_Both_Graph")
data_root = os.path.join(PROJECT_ROOT, f"N_figures/{assets_tag}/PnL_Both_Data")

fig_dir = os.path.join(fig_root, trial_folder)
data_dir = os.path.join(data_root, trial_folder)

os.makedirs(fig_dir, exist_ok=True)
os.makedirs(data_dir, exist_ok=True)

print("\nSaving under:")
print("  fig_dir :", fig_dir)
print("  data_dir:", data_dir)

# ============================================================
# 5. ENV SETTINGS
# ============================================================
terminal_time = 1.0
n_steps = 100
num_assets = int(assets_tag.replace("Assets", ""))
num_trajectories = 1000

train_seed = 123
eval_seed = 456

eval_phi = 0.0
eval_alpha = 0.2

n_eval_episodes = 100000
deterministic = True

# ============================================================
# 6. DYNAMICS (CRITICAL)
# ============================================================
cross_asset_influence = [[0.0, 5.0],
                         [5.0, 0.0]]

c_cross_asset_influence = [[0.0, 3.0],
                           [3.0, 0.0]]

env_kwargs = dict(
    cross_asset_influence=cross_asset_influence,
    c_cross_asset_influence=c_cross_asset_influence,
)

# ============================================================
# 7. BUILD ENV
# ============================================================
set_env_globals(
    terminal_time=terminal_time,
    n_steps=n_steps,
    phi=eval_phi,
    alpha=eval_alpha,
    train_seed=train_seed,
)

train_bundle = make_sb_env(
    get_env_fn=get_cj_env,
    num_trajectories=num_trajectories,
    num_assets=num_assets,
    do_reward_scaling=False,
    env_seed=train_seed,
    **env_kwargs,
)

eval_bundle = make_eval_sb_env(
    get_env_fn=get_cj_env,
    train_bundle=train_bundle,
    eval_seed=eval_seed,
    **env_kwargs,
)

sb_eval_env = VecMonitor(eval_bundle.sb_eval_env)

# ============================================================
# 8. LOAD MODEL
# ============================================================
def load_model_robust(path, env):
    try:
        return PPO.load(path, env=env, device="cpu")
    except:
        return PPO.load(
            path,
            env=env,
            device="cpu",
            custom_objects={
                "learning_rate": 0.0,
                "lr_schedule": lambda _: 0.0,
                "clip_range": lambda _: 0.2,
            },
        )

model = load_model_robust(model_path, sb_eval_env)

# ============================================================
# 9. COLLECT REWARDS
# ============================================================
def collect_rewards(model, env, n):
    rewards = []
    obs = env.reset()

    while len(rewards) < n:
        action, _ = model.predict(obs, deterministic=True)
        obs, _, dones, infos = env.step(action)

        for d, info in zip(dones, infos):
            if d:
                rewards.append(float(info["episode"]["r"]))

    return np.array(rewards[:n])

episode_rewards = collect_rewards(model, sb_eval_env, n_eval_episodes)

# ============================================================
# 10. SUMMARY
# ============================================================
summary = {
    "mean": float(np.mean(episode_rewards)),
    "std": float(np.std(episode_rewards, ddof=1)),
    "variance": float(np.var(episode_rewards, ddof=1)),
    "sharpe": float(np.mean(episode_rewards) / np.std(episode_rewards, ddof=1)),
    "q05": float(np.quantile(episode_rewards, 0.05)),
    "q50": float(np.quantile(episode_rewards, 0.50)),
    "q95": float(np.quantile(episode_rewards, 0.95)),
}

print("\n===== SUMMARY =====")
for k, v in summary.items():
    print(f"{k}: {v:.6f}")

# ============================================================
# 11. SAVE DATA
# ============================================================
base_name = f"{run_tag}_PnL"

np.savetxt(os.path.join(data_dir, f"{base_name}.csv"), episode_rewards)

with open(os.path.join(data_dir, f"{base_name}_summary.json"), "w") as f:
    json.dump(summary, f, indent=2)

# ============================================================
# 12. HISTOGRAM
# ============================================================
plt.figure(figsize=(8,5))
plt.hist(episode_rewards, bins=60)
plt.title("PnL Distribution (rho=5, rho_c=3)")
plt.grid(True)
plt.tight_layout()

plt.savefig(os.path.join(fig_dir, f"{base_name}.pdf"))
plt.savefig(os.path.join(fig_dir, f"{base_name}.png"), dpi=600)

plt.show()

# ============================================================
# DONE
# ============================================================
print("\nDONE — Both coupling evaluation saved.")

In [ ]:
import os
PROJECT_ROOT = os.environ.get("MBT_PROJECT_ROOT")
if not PROJECT_ROOT:
    PROJECT_ROOT = os.getcwd()
    while not os.path.isdir(os.path.join(PROJECT_ROOT, "mbt_gym")) and os.path.dirname(PROJECT_ROOT) != PROJECT_ROOT:
        PROJECT_ROOT = os.path.dirname(PROJECT_ROOT)


# ============================================================
# BOTH COUPLING (MO and LOB) COMPARISON
# Here already evaluated Models being compared (using saved data)
# Basically 3 Distributions of Terminal PnL being stacked (plotted) together.
# Compare:
#   Trial10 → (rho=5, rho_c=0)
#   Trial15 → (rho=0, rho_c=5)
#   Trial18 → (rho=5, rho_c=3)
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# GLOBAL STYLE (MATCH FIRST SCRIPT)
# ============================================================
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman", "CMU Serif", "DejaVu Serif"],
    "mathtext.fontset": "cm",
    "axes.labelsize": 26,
    "xtick.labelsize": 20,
    "ytick.labelsize": 20,
    "axes.linewidth": 1.2,
    "lines.linewidth": 6,
})

# ============================================================
# 1. PATHS
# ============================================================
assets_tag = "2Assets"

data_root_A = os.path.join(PROJECT_ROOT, f"N_figures/{assets_tag}/PnL_Data")
data_root_B = os.path.join(PROJECT_ROOT, f"N_figures/{assets_tag}/PnL_LOB_Data")
data_root_C = os.path.join(PROJECT_ROOT, f"N_figures/{assets_tag}/PnL_Both_Data")

save_root = os.path.join(PROJECT_ROOT, f"N_figures/{assets_tag}/PnL_Both_Comparison")
os.makedirs(save_root, exist_ok=True)

# ============================================================
# 2. CONFIGS
# ============================================================
configs = [
    {
        "label": r"$\rho=5,\ \rho_c=0$",
        "trial": "PPO_2Assets_Trial10",
        "root": data_root_A
    },
    {
        "label": r"$\rho=0,\ \rho_c=5$",
        "trial": "PPO_2Assets_Trial16",
        "root": data_root_B
    },
    {
        "label": r"$\rho=5,\ \rho_c=3$",
        "trial": "PPO_2Assets_Trial18",
        "root": data_root_C
    },
]

# ============================================================
# 3. ROBUST LOADER
# ============================================================
def load_rewards(file_path):
    try:
        return np.loadtxt(file_path)
    except:
        return np.loadtxt(file_path, delimiter=",", skiprows=1)

# ============================================================
# 4. LOAD DATA
# ============================================================
all_rewards = []
summary_rows = []

print("\n===== SUMMARY =====\n")

for cfg in configs:

    trial = cfg["trial"]
    root = cfg["root"]

    data_dir = os.path.join(root, trial)

    if not os.path.exists(data_dir):
        raise FileNotFoundError(f"Missing folder: {data_dir}")

    # FIXED FILE SELECTION (robust for all pipelines)
    files = sorted(
        [f for f in os.listdir(data_dir)
         if (f.endswith(".txt") or f.endswith(".csv"))
         and "summary" not in f.lower()]
    )

    if len(files) == 0:
        raise FileNotFoundError(f"No valid reward file in {data_dir}")

    file_path = os.path.join(data_dir, files[0])

    rewards = load_rewards(file_path)

    mean = np.mean(rewards)
    std = np.std(rewards, ddof=1)
    var = std**2
    sharpe = mean / std if std > 0 else np.nan

    all_rewards.append(rewards)

    summary_rows.append({
        "trial": trial,
        "label": cfg["label"],
        "mean": mean,
        "std": std,
        "variance": var,
        "sharpe": sharpe if not np.isnan(sharpe) else None,
        "n": len(rewards)
    })

    print(f"{cfg['label']}")
    print(f"mean   = {mean:.8f}")
    print(f"std    = {std:.8f}")
    print(f"sharpe = {sharpe:.8f}\n")

summary_df = pd.DataFrame(summary_rows)

# ============================================================
# 5. SAVE SUMMARY
# ============================================================
csv_path = os.path.join(save_root, "PnL_both_comparison_summary.csv")
json_path = os.path.join(save_root, "PnL_both_comparison_summary.json")

summary_df.to_csv(csv_path, index=False)
summary_df.to_json(json_path, orient="records", indent=4)

print("Saved summary:")
print(" ", csv_path)
print(" ", json_path)

# ============================================================
# 6. COMMON BINS (MATCH FIRST SCRIPT)
# ============================================================
global_min = min(np.min(x) for x in all_rewards)
global_max = max(np.max(x) for x in all_rewards)

n_bins = 80
bins = np.linspace(global_min, global_max, n_bins + 1)

# slight tightening only (same as first code)
#margin = 0.02 * (global_max - global_min)
margin = 0.00 * (global_max - global_min)

# ============================================================
# 7. HISTOGRAM OVERLAY (UPDATED STYLE)
# ============================================================
plt.figure(figsize=(10, 6))

for rewards, cfg in zip(all_rewards, configs):
    plt.hist(
        rewards,
        bins=bins,
        alpha=0.35,
        density=True,
        label=cfg["label"]
    )

plt.xlabel("Terminal PnL")
plt.ylabel("Density ")
plt.legend(fontsize=16)

plt.grid(True, linestyle="--", linewidth=0.8, alpha=0.7)
plt.tick_params(width=1.5, length=6)

plt.xlim(global_min + margin, global_max - margin)

plt.tight_layout()

pdf_path = os.path.join(save_root, "PnL_both_hist_overlay.pdf")
png_path = os.path.join(save_root, "PnL_both_hist_overlay.png")

plt.savefig(pdf_path, bbox_inches="tight")
plt.savefig(png_path, dpi=600, bbox_inches="tight")

plt.show()

# ============================================================
# 8. LINE HISTOGRAM (UPDATED STYLE)
# ============================================================
plt.figure(figsize=(10, 6))

for rewards, cfg in zip(all_rewards, configs):

    counts, edges = np.histogram(rewards, bins=bins, density=True)
    centers = 0.5 * (edges[:-1] + edges[1:])

    plt.plot(
        centers,
        counts,
        label=cfg["label"]
    )

plt.xlabel("Terminal PnL")
plt.ylabel("Density")
plt.legend(fontsize=16)

plt.grid(True, linestyle="--", linewidth=0.8, alpha=0.7)
plt.tick_params(width=1.5, length=6)

plt.xlim(global_min + margin, global_max - margin)

plt.tight_layout()

pdf_path = os.path.join(save_root, "PnL_both_hist_lines.pdf")
png_path = os.path.join(save_root, "PnL_both_hist_lines.png")

plt.savefig(pdf_path, bbox_inches="tight")
plt.savefig(png_path, dpi=600, bbox_inches="tight")

plt.show()

# ============================================================
# DONE
# ============================================================
print("\nDONE — Both coupling comparison saved.")